# LatentFed — server GPU batch run: all official seeds × all requested alphas

This version is configured for a **single unattended server run** on one CUDA GPU.

1. The official seeds are fixed to **42, 101, 2024, 3407, 8888**.
2. The requested Dirichlet settings are fixed to **alpha = 0.3, 0.5, 0.7**.
3. The current notebook remains configured for **CIFAR-10**, so one `Run All` executes **15 final experiments** (5 seeds × 3 alpha values), each for 100 communication rounds.
4. Results are checkpointed every five rounds and after every completed experiment under `latentfed_revision_outputs/`.
5. If the process is interrupted, rerun the notebook from the same working directory: the local checkpoint is detected automatically and completed experiments are skipped.
6. `RESUME_FROM` can optionally point to a checkpoint stored elsewhere.
7. CIFAR-10 is downloaded/cached automatically under the local output directory if it is not already available.

The method configuration is frozen before final testing. Final metrics are aggregated across all five seeds using the existing rounds 91–100 protocol and sample standard deviation.


In [ ]:
# Official final-test grid. One Run All executes the whole grid.
FINAL_SEEDS = [42, 101, 2024, 3407, 8888]

# Optional: path to an existing restart bundle stored elsewhere.
# Leave None to start fresh or automatically resume the checkpoint
# already present in OUTPUT_DIR.
RESUME_FROM = None

# All server-side outputs, checkpoints and downloaded data are kept here.
OUTPUT_DIR = "./latentfed_revision_outputs"

print(
    f"Batch configuration: {len(FINAL_SEEDS)} seeds "
    f"({', '.join(map(str, FINAL_SEEDS))}). "
    "The alpha grid and dataset list are defined in the central configuration cell."
)


In [ ]:
import os
# Must be set before the first CUDA operation, including strict GEMM in pooling.
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import sys
import subprocess
import importlib.util


def ensure_package(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or import_name])


# Use the server's CUDA-enabled torch/torchvision installation; do not replace it automatically.
if any(importlib.util.find_spec(name) is None for name in ("torch", "torchvision")):
    raise RuntimeError("The server environment must provide CUDA-enabled torch and torchvision.")
ensure_package("scipy")
ensure_package("sklearn", "scikit-learn")
ensure_package("pandas")
ensure_package("hdbscan")


## 1. Central configuration and execution controls


In [ ]:
from dataclasses import replace
try:
    from IPython.display import display
except ImportError:
    display = print
FAIR_PROTOCOL_VERSION = "neurocomp-fair-v1"
COMMON_PROTOCOL_SHA256 = "445d12940d051634680727dc3739822c4030a90d7a3a3a90894f1d42e490d2fe"
METHOD_IMPLEMENTATION_VERSION = "latentfed-neurocomputing-revision-v2-deterministic-pool"

import copy
import hashlib
import importlib.metadata
import json
import os
import random
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import (adjusted_rand_score, silhouette_score,
                             calinski_harabasz_score, davies_bouldin_score)
from sklearn.metrics.pairwise import cosine_similarity
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.transforms import functional as TF
from tqdm.auto import tqdm

try:
    import hdbscan
except Exception:
    hdbscan = None



CALD_REVISION_VERSION = METHOD_IMPLEMENTATION_VERSION


@dataclass
class ExperimentConfig:
    dataset: str = "cifar10"  # "cifar10" or "emnist"
    alpha: float | None = 0.3  # None means IID split
    method: str = "latentfed_fedavg"  # fedavg, fedprox, latentfed_fedavg, latentfed_fedprox
    seed: int = 42
    num_clients: int = 20
    participation_ratio: float = 1.0
    rounds: int = 100
    local_epochs: int = 1
    batch_size: int = 32
    num_workers: int = 2
    latent_dim: int = 128
    classifier_lr_cifar10: float = 5e-4
    classifier_lr_emnist: float = 1e-4
    autoencoder_lr: float = 5e-4
    weight_decay: float = 1e-4
    lambda_rec: float = 1.0
    lambda_str: float = 0.05
    lambda_inv: float = 0.05
    lambda_clu: float = 0.05
    lambda_margin: float = 0.05
    margin_cls: float = 1.0
    margin_clu: float = 1.0
    fedprox_mu: float = 0.01
    cald_warmup_rounds: int = 20
    cald_ramp_rounds: int = 20
    mdfe_mode: str = "full"  # full, off, mean_only, peak_only, mean_peak, fixed_distortion, std_distortion
    cald_mode: str = "fixed_split"  # adaptive remains a validation comparator
    cald_target: str = "sample"  # sample or batch_mean (separate ablation)
    invariant_ratio: float = 0.5  # initial/hard allocation; learned gate may change it
    router_temperature: float = 1.0
    router_init_logit: float = 2.0
    lambda_route_soft: float = 1e-3  # preserved adaptive control
    lambda_route_budget: float = 1.0
    router_min_fraction: float = 0.25
    router_max_fraction: float = 0.75
    summary_view: str = "deterministic"  # common protocol uses deterministic training-only summaries
    exclude_noise_from_prototypes: bool = True
    dp_noise_std: float = 0.0  # noise perturbation, not a formal DP guarantee
    cluster_min_size: int = 2
    checkpoint_every: int = 5
    cluster_geometry: str = "full"
    cluster_alignment: str = "greedy"
    alignment_max_cosine_distance: float = 1.0  # reject negative cosine similarity
    prototype_ema_beta: float = 0.0
    summary_ema_beta: float = 0.0
    mdfe_rho_mode: str = "positive"
    ae_pretrain_rounds: int = 0
    cluster_min_samples: int = 1
    data_root: str = "./latentfed_revision_outputs/data"
    validation_fraction: float = 0.1
    min_train_samples: int = 10
    min_validation_samples: int = 1
    min_test_samples: int = 1
    partition_max_attempts: int = 100
    evaluation_split: str = "validation"  # set validation during hyperparameter selection
    allow_custom_protocol: bool = False

    @property
    def uses_latentfed(self) -> bool:
        return self.method.startswith("latentfed")

    @property
    def uses_fedprox(self) -> bool:
        return self.method in {"fedprox", "latentfed_fedprox"}

    @property
    def classifier_lr(self) -> float:
        return self.classifier_lr_cifar10 if self.dataset == "cifar10" else self.classifier_lr_emnist

    def validate(self):
        validate_common_config(self)
        if self.cluster_geometry not in {"full", "cluster_only"} or self.cluster_alignment not in {"greedy", "hungarian"}:
            raise ValueError("Invalid geometry/alignment")
        if self.mdfe_rho_mode not in {"original", "positive"}:
            raise ValueError("Invalid rho mode")
        if not all(0 <= b < 1 for b in (self.prototype_ema_beta, self.summary_ema_beta)):
            raise ValueError("EMA beta must be in [0, 1)")
        if not 0 <= self.ae_pretrain_rounds < self.rounds:
            raise ValueError("Warm-up must fit inside the fixed AE update budget")
        if self.cald_mode == "fixed_split" and (self.latent_dim % 2 or self.invariant_ratio != .5):
            raise ValueError("Paper fixed split requires equal halves")
        if self.cluster_min_size < 2 or self.cluster_min_samples < 1 or not 0 <= self.alignment_max_cosine_distance <= 1:
            raise ValueError("Invalid clustering parameters")

        if self.mdfe_mode not in {"full", "off", "mean_only", "peak_only", "mean_peak", "fixed_distortion", "std_distortion"}:
            raise ValueError("Unknown mdfe_mode")
        if self.cald_mode not in {"adaptive", "fixed_split", "fixed_soft", "legacy", "off"}:
            raise ValueError("Unknown cald_mode")
        if self.cald_target not in {"sample", "batch_mean"}:
            raise ValueError("cald_target must be sample or batch_mean")
        if self.summary_view not in {"train", "deterministic"}:
            raise ValueError("summary_view must be train or deterministic")
        if self.latent_dim < 2 or not 0 < self.invariant_ratio < 1 or not 0 < self.invariant_dim < self.latent_dim:
            raise ValueError("Both latent branches must have at least one coordinate")
        if self.cald_mode == "legacy" and (self.latent_dim % 2 or self.invariant_ratio != .5 or self.cald_target != "sample"):
            raise ValueError("legacy requires an even latent_dim, ratio 0.5 and sample targets")
        if self.router_temperature <= 0 or self.router_init_logit <= 0:
            raise ValueError("Router temperature and initial logit magnitude must be positive")
        if not 0 < self.router_min_fraction < self.router_max_fraction < 1:
            raise ValueError("Invalid routing budget range")
        if min(self.lambda_route_soft, self.lambda_route_budget, self.dp_noise_std, self.cald_warmup_rounds, self.cald_ramp_rounds) < 0:
            raise ValueError("Loss weights, noise and curriculum durations must be nonnegative")
        if not 0 < self.participation_ratio <= 1 or min(self.rounds, self.local_epochs, self.batch_size, self.num_clients, self.checkpoint_every) < 1:
            raise ValueError("Invalid training budget or participation")
        if self.uses_latentfed and hdbscan is None:
            raise RuntimeError("HDBSCAN is required. Install hdbscan and restart; no silent DBSCAN substitution is allowed.")

    @property
    def invariant_dim(self):
        return int(round(self.latent_dim * self.invariant_ratio))

    def set_scenario(self, scenario):
        profiles = {"20_full": (20, 1.0), "100_random10": (100, 0.1)}
        if scenario not in profiles:
            raise ValueError(f"Unknown scenario: {scenario}")
        self.num_clients, self.participation_ratio = profiles[scenario]
        return self

    @property
    def config_id(self):
        excluded = {"data_root", "checkpoint_every", "num_workers"}
        payload = {k: v for k, v in asdict(self).items() if k not in excluded}
        payload.update(protocol_version=FAIR_PROTOCOL_VERSION, common_code=COMMON_PROTOCOL_SHA256,
                       method_code=METHOD_IMPLEMENTATION_VERSION)
        return hashlib.sha256(json.dumps(payload, sort_keys=True, allow_nan=False).encode()).hexdigest()[:12]

    @property
    def run_name(self):
        dist = "iid" if self.alpha is None else f"alpha_{self.alpha}"
        return f"{self.dataset}_{dist}_{self.method}_n{self.num_clients}_p{self.participation_ratio:g}_{self.evaluation_split}_seed{self.seed}_{self.config_id}"




DEV_SEEDS = [7, 19]
SCENARIO = "20_full"
CLUSTER_GEOMETRY = "full"       # alternatives: full, cluster_only
CALD_MODE = "fixed_split"      # paper default; adaptive remains a dev comparator
CLUSTER_ALIGNMENT = "greedy"   # Hungarian must earn promotion on validation
PROTOTYPE_EMA_BETA = 0.0
MDFE_RHO_MODE = "positive"     # correctness constraint, original is a diagnostic control
AE_PRETRAIN_ROUNDS = 0
CLIENT_SUMMARY_EMA_BETA = 0.0
CFG = ExperimentConfig(seed=7, cluster_geometry=CLUSTER_GEOMETRY, cald_mode=CALD_MODE,
    cluster_alignment=CLUSTER_ALIGNMENT, prototype_ema_beta=PROTOTYPE_EMA_BETA,
    mdfe_rho_mode=MDFE_RHO_MODE, ae_pretrain_rounds=AE_PRETRAIN_ROUNDS,
    summary_ema_beta=CLIENT_SUMMARY_EMA_BETA).set_scenario(SCENARIO)
# Final-run settings: one server run executes the complete official grid.
RUN_DEVELOPMENT = False
RUN_FINAL = True
ENABLE_CHECKPOINT = True

OUTPUT_DIR_PATH = Path(OUTPUT_DIR).expanduser().resolve()
OUTPUT_DIR_PATH.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = OUTPUT_DIR_PATH / "latentfed_revision_state.pt"
RESUME_PATH = Path(RESUME_FROM).expanduser().resolve() if RESUME_FROM is not None else CHECKPOINT_PATH
RESUME_BUNDLE = RESUME_FROM is not None or CHECKPOINT_PATH.is_file()

# Keep the dataset scope of the uploaded notebook (CIFAR-10), but run all requested alphas.
FINAL_DATASETS = ["cifar10"]
FINAL_ALPHAS = [0.3, 0.5, 0.7]

EXPORT_PAPER_ZIP = True
PAPER_ZIP_PATH = OUTPUT_DIR_PATH / "latentfed_paper_evaluation_all_seeds_all_alphas.zip"
ZIP_INCLUDE_RESTART = True

print(
    f"Final grid: {len(FINAL_SEEDS)} seeds × {len(FINAL_DATASETS)} dataset(s) × "
    f"{len(FINAL_ALPHAS)} alpha values = "
    f"{len(FINAL_SEEDS) * len(FINAL_DATASETS) * len(FINAL_ALPHAS)} experiments."
)
print(f"Output directory: {OUTPUT_DIR_PATH}")
# Predeclared global selection guardrails; proportions, not percentage points.
SELECTION_RULE = dict(min_wacc_gain=.001, max_macc_drop=.005, max_wc_drop=.01,
                      max_ari_drop=.05, max_noise_increase=.10,
                      min_multicluster_fraction=.5, max_mean_noise=.8)
display(pd.DataFrame([asdict(CFG)]).T.rename(columns={0: "Method settings to freeze before the final test"}))


## 2. Reproducible streams and shared data protocol
Training, held-out validation and test indices remain disjoint. Partition and classifier initialization algorithms are preserved from the source. Only the selected evaluation split is iterated; no model or hyperparameter selection uses test predictions.


In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False


def get_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def now_stamp() -> str:
    return time.strftime("%Y%m%d_%H%M%S")


def safe_torch_load(path: str | Path, map_location: str | torch.device = "cpu") -> dict[str, Any]:
    try:
        return torch.load(str(path), map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(str(path), map_location=map_location)


def state_dict_to_cpu(state: dict[str, torch.Tensor]) -> dict[str, torch.Tensor]:
    return {k: v.detach().cpu().clone() for k, v in state.items()}


def weighted_average_states(states: list[dict[str, torch.Tensor]], weights: list[float]) -> dict[str, torch.Tensor]:
    total = float(np.sum(weights))
    if total <= 0:
        weights = [1.0 / len(states)] * len(states)
    else:
        weights = [float(w) / total for w in weights]

    out: dict[str, torch.Tensor] = {}
    for key in states[0].keys():
        if not torch.is_floating_point(states[0][key]):
            out[key] = states[0][key].detach().cpu().clone()
            continue
        acc = None
        for state, weight in zip(states, weights):
            value = state[key].detach().float().cpu() * weight
            acc = value if acc is None else acc + value
        out[key] = acc
    return out


def l2_normalize_np(x: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    return x / (np.linalg.norm(x, axis=-1, keepdims=True) + eps)


DEVICE = get_device()
set_seed(CFG.seed)
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(torch.cuda.get_device_name(0))


def verify_final_runtime():
    """Fail before long work if the server does not expose a working CUDA GPU."""
    if not torch.cuda.is_available():
        raise RuntimeError("Final tests require a CUDA GPU. Verify the server GPU, NVIDIA driver and CUDA-enabled PyTorch installation.")
    torch.cuda.set_device(0)
    # Exercise strict deterministic GEMM on the actual CUDA runtime.
    probe = torch.ones(4, 4, device="cuda", requires_grad=True)
    (probe @ probe).sum().backward()
    torch.cuda.synchronize()
    if not torch.isfinite(probe.grad).all():
        raise RuntimeError("CUDA preflight returned a nonfinite gradient")
    print(f"Final-test GPU: {torch.cuda.get_device_name(0)}; torch={torch.__version__}; "
          f"CUDA={torch.version.cuda}; strict determinism enabled; device cuda:0.")
    return torch.device("cuda:0")

if RUN_FINAL:
    verify_final_runtime()


In [ ]:
from contextlib import contextmanager
from torch.utils.data import Dataset

class EMNISTFixOrientation:
    def __call__(self, img):
        return TF.hflip(TF.rotate(img, -90))


def get_dataset_bundle(dataset_name: str, data_root: str = "./latentfed_revision_outputs/data"):
    dataset_name = dataset_name.lower()
    if dataset_name == "cifar10":
        train_transform = transforms.Compose(
            [
                transforms.RandomCrop(32, padding=4),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                transforms.Normalize(
                    (0.4914, 0.4822, 0.4465),
                    (0.2470, 0.2435, 0.2616),
                ),
            ]
        )
    
        test_transform = transforms.Compose(
            [
                transforms.ToTensor(),
                transforms.Normalize(
                    (0.4914, 0.4822, 0.4465),
                    (0.2470, 0.2435, 0.2616),
                ),
            ]
        )
    
        # Server-portable CIFAR-10 loading.
        # torchvision reuses the local cache when present and downloads only when needed.
        data_root_path = Path(data_root).expanduser().resolve()
        data_root_path.mkdir(parents=True, exist_ok=True)

        train_ds = datasets.CIFAR10(
            root=str(data_root_path),
            train=True,
            download=True,
            transform=train_transform,
        )

        test_ds = datasets.CIFAR10(
            root=str(data_root_path),
            train=False,
            download=True,
            transform=test_transform,
        )

        labels_train = np.asarray(train_ds.targets)
        labels_test = np.asarray(test_ds.targets)

        print(
            f"CIFAR-10 ready at {data_root_path}: "
            f"{len(train_ds):,} train / {len(test_ds):,} test"
        )

        return train_ds, test_ds, labels_train, labels_test, 3, 32, 10
    if dataset_name == "emnist":
        train_transform = transforms.Compose(
            [
                EMNISTFixOrientation(),
                transforms.ToTensor(),
                transforms.Normalize((0.1307,), (0.3081,)),
            ]
        )
        test_transform = transforms.Compose(
            [
                EMNISTFixOrientation(),
                transforms.ToTensor(),
                transforms.Normalize((0.1307,), (0.3081,)),
            ]
        )
        train_ds = datasets.EMNIST(root=data_root, split="balanced", train=True, download=True, transform=train_transform)
        test_ds = datasets.EMNIST(root=data_root, split="balanced", train=False, download=True, transform=test_transform)
        labels_train = np.asarray(train_ds.targets)
        labels_test = np.asarray(test_ds.targets)
        return train_ds, test_ds, labels_train, labels_test, 1, 28, 47

    raise ValueError(f"Unsupported dataset: {dataset_name}")


def role_seed(seed, *roles):
    value = json.dumps([int(seed), *roles], separators=(",", ":"))
    return int.from_bytes(hashlib.sha256(value.encode()).digest()[:4], "little")


@contextmanager
def cpu_random_stream(seed):
    """Isolate augmentation/model construction from other algorithm activity."""
    python_state, numpy_state = random.getstate(), np.random.get_state()
    try:
        with torch.random.fork_rng(devices=[]):
            random.seed(seed)
            np.random.seed(seed)
            torch.random.default_generator.manual_seed(seed)
            yield
    finally:
        random.setstate(python_state)
        np.random.set_state(numpy_state)


def array_hash(arrays):
    digest = hashlib.sha256()
    for key, value in sorted(arrays.items()):
        a = np.ascontiguousarray(value)
        digest.update(key.encode())
        digest.update(str(a.dtype).encode())
        digest.update(json.dumps(a.shape).encode())
        digest.update(a.tobytes())
    return digest.hexdigest()


def common_selection(cfg, round_idx):
    # Independent of alpha, method, initialization and prior selections.
    count = max(1, int(round(cfg.num_clients * cfg.participation_ratio)))
    rng = np.random.default_rng(role_seed(cfg.seed, "participants", cfg.num_clients, round_idx))
    return sorted(rng.choice(cfg.num_clients, size=count, replace=False).tolist())


def validate_common_config(cfg):
    if not cfg.allow_custom_protocol:
        if (cfg.num_clients, cfg.participation_ratio) not in {(20, 1.0), (100, .1)}:
            raise ValueError("Use set_scenario('20_full') or set_scenario('100_random10'); custom studies require allow_custom_protocol=True")
        expected = {"local_epochs": 1, "batch_size": 32, "classifier_lr_cifar10": 5e-4,
                    "classifier_lr_emnist": 1e-4, "weight_decay": 1e-4}
        for key, value in expected.items():
            if getattr(cfg, key) != value:
                raise ValueError(f"Common benchmark requires {key}={value}; declare a custom protocol for another setting")
        if getattr(cfg, "local_optimizer", "adam").lower() != "adam":
            raise ValueError("The common classifier optimizer is Adam")
    if getattr(cfg, "summary_view", "deterministic") != "deterministic":
        raise ValueError("The common protocol requires a deterministic training summary view")
    if cfg.dataset not in {"cifar10", "emnist"} or cfg.evaluation_split not in {"validation", "test"}:
        raise ValueError("Unknown dataset/evaluation split")
    if not 0 < cfg.validation_fraction < 1 or not 0 < cfg.participation_ratio <= 1:
        raise ValueError("Invalid validation or participation fraction")
    if cfg.alpha is not None and (not np.isfinite(cfg.alpha) or cfg.alpha <= 0):
        raise ValueError("Dirichlet alpha must be positive or None for IID")
    if min(cfg.num_clients, cfg.rounds, cfg.batch_size, cfg.local_epochs, cfg.partition_max_attempts,
           cfg.min_train_samples, cfg.min_validation_samples, cfg.min_test_samples,
           cfg.checkpoint_every) < 1:
        raise ValueError("Training budgets and partition minimums must be positive")
    if cfg.participation_ratio < 1 and getattr(cfg, "reassign_all_clients_each_round", False) and cfg.method in {"ifca", "hypcluster"}:
        raise ValueError("Loss-based reassignment may contact only selected clients in the partial-participation protocol")


def allocate_counts(count, probabilities):
    expected = count * probabilities
    counts = np.floor(expected).astype(np.int64)
    order = np.argsort(-(expected - counts), kind="stable")
    counts[order[:count - counts.sum()]] += 1
    return counts


def make_common_partitions(train_labels, test_labels, cfg):
    """One class-to-client probability matrix for both official data splits."""
    train_labels = np.asarray(train_labels, dtype=np.int64)
    test_labels = np.asarray(test_labels, dtype=np.int64)
    classes = np.unique(train_labels)
    if not np.array_equal(classes, np.arange(len(classes))) or not np.isin(test_labels, classes).all():
        raise ValueError("Labels must be contiguous zero-based classes, with test classes present in training")
    if len(train_labels) < cfg.num_clients * (cfg.min_train_samples + cfg.min_validation_samples) or len(test_labels) < cfg.num_clients * cfg.min_test_samples:
        raise ValueError("Dataset is too small for the requested client minimums")
    n = cfg.num_clients
    for attempt in range(cfg.partition_max_attempts):
        rng = np.random.default_rng(role_seed(cfg.seed, "allocation", attempt))
        probabilities = np.full((len(classes), n), 1 / n) if cfg.alpha is None else rng.dirichlet(np.full(n, cfg.alpha), size=len(classes))
        groups = {split: [[] for _ in range(n)] for split in ("train", "validation", "test")}
        for cls in classes:
            train_idx = np.where(train_labels == cls)[0]
            test_idx = np.where(test_labels == cls)[0]
            rng.shuffle(train_idx)
            rng.shuffle(test_idx)
            train_chunks = np.split(train_idx, np.cumsum(allocate_counts(len(train_idx), probabilities[cls]))[:-1])
            test_chunks = np.split(test_idx, np.cumsum(allocate_counts(len(test_idx), probabilities[cls]))[:-1])
            for client, (tr, te) in enumerate(zip(train_chunks, test_chunks)):
                # Stratify within each client's class; retain one training example.
                nv = min(max(1, int(round(len(tr) * cfg.validation_fraction))), len(tr) - 1) if len(tr) >= 2 else 0
                groups["validation"][client].extend(tr[:nv].tolist())
                groups["train"][client].extend(tr[nv:].tolist())
                groups["test"][client].extend(te.tolist())
        minimums = {"train": cfg.min_train_samples, "validation": cfg.min_validation_samples, "test": cfg.min_test_samples}
        if all(min(map(len, groups[s])) >= minimums[s] for s in groups):
            arrays = {f"{split}_{i}": np.asarray(rng.permutation(ids), dtype=np.int64)
                      for split, clients in groups.items() for i, ids in enumerate(clients)}
            arrays["allocation"] = probabilities
            return arrays, attempt + 1
    raise ValueError("Partition minimums were not met; no invalid/empty-client split was returned")


def verify_partitions(arrays, train_labels, test_labels, cfg):
    allocation = arrays["allocation"]
    if allocation.shape != (len(np.unique(train_labels)), cfg.num_clients) or not np.isfinite(allocation).all() or (allocation < 0).any() or not np.allclose(allocation.sum(axis=1), 1):
        raise ValueError("Invalid shared allocation probabilities")
    for splits, labels in [(("train", "validation"), train_labels), (("test",), test_labels)]:
        indices = np.concatenate([arrays[f"{s}_{i}"] for s in splits for i in range(cfg.num_clients)])
        if not np.array_equal(np.sort(indices), np.arange(len(labels))):
            raise ValueError("Partitions have missing, duplicate or out-of-range indices")
    for split, minimum in [("train", cfg.min_train_samples), ("validation", cfg.min_validation_samples), ("test", cfg.min_test_samples)]:
        if min(len(arrays[f"{split}_{i}"]) for i in range(cfg.num_clients)) < minimum:
            raise ValueError("Cached client minimum is invalid")


class PairedTrainingView(Dataset):
    def __init__(self, dataset, indices, seed, client, round_idx, epoch):
        self.dataset, self.indices = dataset, list(map(int, indices))
        self.seed, self.client, self.round_idx, self.epoch = seed, client, round_idx, epoch

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        raw_index = self.indices[index]
        with cpu_random_stream(role_seed(self.seed, "augmentation", self.client, self.round_idx, self.epoch, raw_index)):
            return self.dataset[raw_index]


class PairedClientLoader:
    """Private per-client/round/epoch shuffle and augmentation streams."""
    def __init__(self, dataset, indices, cfg, client):
        self.source, self.indices, self.cfg, self.client = dataset, indices, cfg, client
        self.dataset = Subset(dataset, indices.tolist())
        self.set_round(0)

    def set_round(self, round_idx):
        self.round_idx, self.epoch = round_idx, 0

    def __iter__(self):
        epoch = self.epoch
        self.epoch += 1
        view = PairedTrainingView(self.source, self.indices, self.cfg.seed, self.client, self.round_idx, epoch)
        generator = torch.Generator().manual_seed(role_seed(self.cfg.seed, "minibatches", self.client, self.round_idx, epoch))
        return iter(DataLoader(view, batch_size=self.cfg.batch_size, shuffle=True,
                    num_workers=self.cfg.num_workers, generator=generator,
                    pin_memory=torch.cuda.is_available(), drop_last=False))

    def __len__(self):
        return (len(self.indices) + self.cfg.batch_size - 1) // self.cfg.batch_size


def deterministic_loaders(dataset, parts, cfg):
    return [DataLoader(Subset(dataset, ids.tolist()), batch_size=cfg.batch_size * 4,
                       shuffle=False, num_workers=cfg.num_workers,
                       generator=torch.Generator().manual_seed(role_seed(cfg.seed, "evaluation-loader", i)),
                       pin_memory=torch.cuda.is_available()) for i, ids in enumerate(parts)]


def isolated_model_state(factory, cfg, role):
    with cpu_random_stream(role_seed(cfg.seed, role, cfg.dataset)):
        model = factory()
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


class FairRunData:
    """Preserved partition/initialization, with metadata held in memory."""
    def __init__(self, cfg, bundle):
        train, test, tr_y, te_y, channels, size, classes = bundle
        arrays, attempts = make_common_partitions(tr_y, te_y, cfg)
        verify_partitions(arrays, tr_y, te_y, cfg)
        self.arrays = arrays
        self.train_parts, self.validation_parts, self.test_parts = [
            [arrays[f"{s}_{i}"] for i in range(cfg.num_clients)] for s in ("train", "validation", "test")]
        self.base_state = isolated_model_state(lambda: SimpleCNN(channels, classes), cfg, "classifier-initialization")
        view = copy.copy(train)
        view.transform = test.transform
        self.train_loaders = [PairedClientLoader(train, ids, cfg, i) for i, ids in enumerate(self.train_parts)]
        self.assignment_loaders = deterministic_loaders(view, self.train_parts, cfg)
        self.evaluation_loaders = deterministic_loaders(
            test if cfg.evaluation_split == "test" else view,
            self.test_parts if cfg.evaluation_split == "test" else self.validation_parts, cfg)
        self.schedule = [common_selection(cfg, r) for r in range(1, cfg.rounds + 1)]
        self.manifest = dict(config=asdict(cfg), partition_hash=array_hash(arrays), partition_attempts=attempts,
            initial_state_hash=array_hash({k: v.numpy() for k, v in self.base_state.items()}),
            schedule_hash=array_hash({"participants": np.asarray(self.schedule)}),
            packages={p: importlib.metadata.version(p) for p in ("torch", "torchvision", "numpy", "pandas", "scipy", "scikit-learn", "hdbscan")},
            device=str(get_device()), cuda=torch.version.cuda,
            gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
            deterministic_algorithms=True, classifier_pooling="equivalent_linear_adaptive_average_v1")

    def set_round(self, round_idx):
        for loader in self.train_loaders:
            loader.set_round(round_idx)


## 3. Unchanged CNN, MDFE and CALD
`fixed_split` returns the first/second 64 coordinates. `adaptive` retains the existing learned 128-coordinate masks. Each CALD branch is normalized independently. Positive rho uses softplus with effective initial value 1; monotonicity concerns the distortion path with the base descriptor term held fixed.

The existing squared class-separation hinge is preserved for controlled comparisons; manuscript Eq. (4) currently shows an unsquared hinge and must be reconciled.

The classifier retains the same adaptive-average-pooling operation, output shape and parameters. Its implementation uses an equivalent fixed averaging matrix (identity for CIFAR's 4×4 map, overlapping bins for EMNIST's 3×3 map), because the native CUDA backward raises under strict determinism. This preserves `torch.use_deterministic_algorithms(True)`; arithmetic is equivalent within floating-point tolerance, not guaranteed bit-identical to the native kernel. [PyTorch deterministic-operation documentation](https://docs.pytorch.org/docs/stable/generated/torch.use_deterministic_algorithms.html).


In [ ]:
class DeterministicAdaptiveAvgPool2d(nn.AdaptiveAvgPool2d):
    """Same adaptive averaging bins and state_dict as nn.AdaptiveAvgPool2d.

    Avoid its unsupported deterministic CUDA backward: express the fixed linear
    average as F.linear. No trainable weights, RNG calls or model-state changes.
    CUBLAS_WORKSPACE_CONFIG is set in the dependency cell for strict CUDA matmul.
    """
    def forward(self, x):
        if x.ndim not in (3, 4):
            raise ValueError("AdaptiveAvgPool2d expects CHW or NCHW input")
        height, width = x.shape[-2:]
        target = (self.output_size, self.output_size) if isinstance(self.output_size, int) else self.output_size
        out_h = height if target[0] is None else target[0]
        out_w = width if target[1] is None else target[1]
        if min(height, width, out_h, out_w) < 1:
            raise ValueError("Pooling dimensions must be positive")
        if (height, width) == (out_h, out_w):
            return x  # CIFAR-10: 4x4 -> 4x4 is exactly the identity.

        def bin_weights(input_size, output_size):
            output = torch.arange(output_size, device=x.device)
            start = torch.div(output * input_size, output_size, rounding_mode="floor")
            end = torch.div((output + 1) * input_size + output_size - 1,
                            output_size, rounding_mode="floor")
            positions = torch.arange(input_size, device=x.device)
            mask = (positions[None, :] >= start[:, None]) & (positions[None, :] < end[:, None])
            return mask.to(x.dtype) / (end - start).to(x.dtype)[:, None]

        rows, columns = bin_weights(height, out_h), bin_weights(width, out_w)
        weights = (rows[:, None, :, None] * columns[None, :, None, :]).reshape(out_h * out_w, height * width)
        # EMNIST uses 3x3 -> 4x4, including the original overlapping averaging bins.
        return F.linear(x.flatten(-2), weights).reshape(*x.shape[:-2], out_h, out_w)


class SimpleCNN(nn.Module):
    def __init__(self, in_channels: int, num_classes: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            DeterministicAdaptiveAvgPool2d((4, 4)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.features(x))


class MDFEGate(nn.Module):
    """Channel gate; full preserves the pilot formula and initialization exactly."""
    MODES = {"full", "off", "mean_only", "peak_only", "mean_peak", "fixed_distortion", "std_distortion"}

    def __init__(self, channels: int, reduction: int = 4):
        super().__init__()
        hidden = max(8, channels // reduction)
        self.base = nn.Sequential(
            nn.Linear(channels * 2, hidden), nn.ReLU(inplace=True), nn.Linear(hidden, channels))
        self.rho_raw = nn.Parameter(torch.ones(channels))
        self.rho_mode = "original"
        self.zeta = nn.Parameter(torch.zeros(channels))
        self.gamma = nn.Parameter(torch.tensor(1.0))
        self.mode = "full"

    def configure_rho(self, mode):
        self.rho_mode = mode
        with torch.no_grad():
            self.rho_raw.fill_(float(np.log(np.expm1(1.0))) if mode == "positive" else 1.0)

    def effective_rho(self):
        return F.softplus(self.rho_raw) if self.rho_mode == "positive" else self.rho_raw

    def configure_mode(self, mode):
        if mode not in self.MODES:
            raise ValueError(f"Unknown MDFE mode: {mode}")
        self.mode = mode
        # Keep the same allocated parameters and RNG consumption in every control.
        for parameter in self.base.parameters():
            parameter.requires_grad_(mode != "off")
        for parameter in (self.rho_raw, self.zeta, self.gamma):
            parameter.requires_grad_(mode in {"full", "std_distortion"})

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.mode == "off":
            return x
        mean_act = x.mean(dim=(2, 3))
        peak_act = x.amax(dim=(2, 3))
        if self.mode in {"mean_only", "peak_only", "mean_peak"}:
            mean_input = torch.zeros_like(mean_act) if self.mode == "peak_only" else mean_act
            peak_input = torch.zeros_like(peak_act) if self.mode == "mean_only" else peak_act
            gate = torch.sigmoid(self.base(torch.cat([mean_input, peak_input], dim=1)))
        else:
            distortion = peak_act - mean_act
            if self.mode == "std_distortion":
                # A variance-derived statistic in activation units; epsilon keeps
                # gradients finite for constant/ReLU-zero feature maps.
                distortion = torch.sqrt(x.var(dim=(2, 3), unbiased=False) + 1e-8)
            base_gate = self.base(torch.cat([mean_act, peak_act], dim=1))
            if self.mode == "fixed_distortion":
                # rho=1, zeta=0, raw gamma=1, hence effective gamma=softplus(1).
                dist_gate = torch.sigmoid(distortion)
                gamma = F.softplus(x.new_tensor(1.0))
            else:
                dist_gate = torch.sigmoid(distortion * self.effective_rho() + self.zeta)
                gamma = F.softplus(self.gamma)
            gate = torch.sigmoid(base_gate - gamma * dist_gate)
        return x * gate[:, :, None, None]


def mdfe_parameter_diagnostics(ae_state, cfg):
    raw = ae_state["mdfe.rho_raw"]
    rho = F.softplus(raw) if cfg.mdfe_rho_mode == "positive" else raw
    return {"mdfe_rho_min": float(rho.min()), "mdfe_rho_mean": float(rho.mean()),
            "mdfe_rho_max": float(rho.max()), "mdfe_rho_negative_fraction": float((rho < 0).float().mean()),
            "mdfe_zeta_mean": float(ae_state["mdfe.zeta"].mean()),
            "mdfe_gamma_effective": float(F.softplus(ae_state["mdfe.gamma"])),
            "mdfe_distortion_active": int(cfg.mdfe_mode in {"full", "fixed_distortion", "std_distortion"})}


class MDFEAutoencoder(nn.Module):
    def __init__(self, in_channels: int, image_size: int, latent_dim: int):
        super().__init__()
        self.encoder_conv = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(inplace=True),
        )
        self.mdfe = MDFEGate(64)
        with torch.no_grad():
            dummy = torch.zeros(1, in_channels, image_size, image_size)
            feat = self.mdfe(self.encoder_conv(dummy))
        self.feature_shape = tuple(feat.shape[1:])
        flat_dim = int(np.prod(self.feature_shape))
        self.to_latent = nn.Linear(flat_dim, latent_dim)
        self.from_latent = nn.Linear(latent_dim, flat_dim)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(32, in_channels, kernel_size=4, stride=2, padding=1),
        )

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        feat = self.mdfe(self.encoder_conv(x))
        return self.to_latent(torch.flatten(feat, start_dim=1))

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        feat = self.from_latent(z).view(z.shape[0], *self.feature_shape)
        return self.decoder(feat)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        z = self.encode(x)
        recon = self.decode(z)
        return recon, z


def class_structured_loss(z: torch.Tensor, y: torch.Tensor, margin: float) -> torch.Tensor:
    unique_labels = torch.unique(y)
    if len(unique_labels) == 0:
        return z.new_tensor(0.0)

    prototypes = []
    compactness_terms = []
    for cls in unique_labels:
        mask = y == cls
        z_cls = z[mask]
        proto = z_cls.mean(dim=0)
        prototypes.append(proto)
        compactness_terms.append(((z_cls - proto) ** 2).sum(dim=1).mean())

    compactness = torch.stack(compactness_terms).mean()
    if len(prototypes) < 2:
        return compactness

    proto = torch.stack(prototypes)
    distances = torch.pdist(proto, p=2)
    separation = F.relu(margin - distances).pow(2).mean()
    return compactness + separation


def cald_curriculum(round_idx: int, cfg: ExperimentConfig) -> float:
    if round_idx <= cfg.cald_warmup_rounds:
        return 0.0
    ramp_pos = round_idx - cfg.cald_warmup_rounds
    return min(1.0, ramp_pos / max(1, cfg.cald_ramp_rounds))


class CALDRouter(nn.Module):
    """Shared coordinate allocation; only adaptive mode adds trainable weights."""
    def __init__(self, cfg: ExperimentConfig):
        super().__init__()
        self.mode = cfg.cald_mode
        self.split = cfg.invariant_dim
        self.temperature = cfg.router_temperature
        hard = (torch.arange(cfg.latent_dim) < self.split).float()
        self.register_buffer("hard_gate", hard)
        initial = (2 * hard - 1) * cfg.router_init_logit
        if self.mode == "adaptive":
            self.logits = nn.Parameter(initial)
        else:
            self.register_buffer("logits", initial)

    def gate(self) -> torch.Tensor:
        if self.mode in {"fixed_split", "legacy", "off"}:
            return self.hard_gate
        return torch.sigmoid(self.logits / self.temperature)

    def raw_branches(self, z: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        if self.mode == "fixed_split":
            return z[:, :self.split], z[:, self.split:]
        if self.mode == "legacy":
            return torch.chunk(F.normalize(z, dim=1), 2, dim=1)
        g = self.gate()
        return z * g, z * (1 - g)

    def forward(self, z: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        inv, clu = self.raw_branches(z)
        if self.mode == "legacy":
            return inv, clu
        return F.normalize(inv, dim=1, eps=1e-8), F.normalize(clu, dim=1, eps=1e-8)


def cald_loss(
    z: torch.Tensor,
    prev_cluster: int | None,
    prototypes: dict[str, Any],
    round_idx: int,
    cfg: ExperimentConfig,
    router: CALDRouter,
) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
    """Return weighted components whose sum equals the CALD contribution."""
    keys = ("cald_inv", "cald_clu", "cald_margin", "route_soft", "route_budget")
    terms = {key: z.new_zeros(()) for key in keys}
    kappa = cald_curriculum(round_idx, cfg)
    if cfg.cald_mode == "off" or kappa <= 0:
        return z.new_zeros(()), terms
    features = z.mean(dim=0, keepdim=True) if cfg.cald_target == "batch_mean" else z
    inv, clu = router(features)
    raw_inv, raw_clu = router.raw_branches(features)
    valid_inv = raw_inv.norm(dim=1) > 1e-8
    valid_clu = raw_clu.norm(dim=1) > 1e-8

    def valid_target(value, branch):
        if value is None:
            return None
        target = torch.as_tensor(value, dtype=z.dtype, device=z.device).detach()
        if target.shape != branch.shape[1:]:
            raise ValueError("Prototype shape does not match CALD mode; start a compatible run.")
        if not torch.isfinite(target).all():
            raise ValueError("Nonfinite CALD prototype.")
        return target if target.norm() > 1e-8 else None

    target = valid_target(prototypes.get("inv"), inv)
    if target is not None and valid_inv.any():
        terms["cald_inv"] = kappa * cfg.lambda_inv * (inv[valid_inv] - target).square().sum(1).mean()
    cluster_protos = prototypes.get("cluster", {})
    own = valid_target(cluster_protos.get(prev_cluster), clu)
    if own is not None and valid_clu.any():
        terms["cald_clu"] = kappa * cfg.lambda_clu * (clu[valid_clu] - own).square().sum(1).mean()
        repulsion = []
        for cluster_id, value in cluster_protos.items():
            if cluster_id == prev_cluster:
                continue
            other = valid_target(value, clu)
            if other is not None:
                distance = (clu[valid_clu] - other).norm(dim=1)
                repulsion.append(F.relu(cfg.margin_clu - distance).square().mean())
        if repulsion:
            terms["cald_margin"] = kappa * cfg.lambda_margin * torch.stack(repulsion).mean()
    if cfg.cald_mode == "adaptive":
        g = router.gate()
        terms["route_soft"] = kappa * cfg.lambda_route_soft * (g * (1 - g)).mean()
        budget = F.relu(cfg.router_min_fraction - g.mean()).square()
        budget = budget + F.relu(g.mean() - cfg.router_max_fraction).square()
        terms["route_budget"] = kappa * cfg.lambda_route_budget * budget
    return sum(terms.values()), terms


def make_autoencoder(in_channels: int, image_size: int, cfg: ExperimentConfig) -> MDFEAutoencoder:
    model = MDFEAutoencoder(in_channels, image_size, cfg.latent_dim)
    # Added after the old layers: router construction consumes no random numbers.
    model.mdfe.configure_rho(cfg.mdfe_rho_mode)
    model.mdfe.configure_mode(cfg.mdfe_mode)
    model.router = CALDRouter(cfg)
    return model


def server_router(ae_state: dict[str, torch.Tensor], cfg: ExperimentConfig) -> CALDRouter:
    router = CALDRouter(cfg)
    router.load_state_dict({key.removeprefix("router."): value for key, value in ae_state.items() if key.startswith("router.")})
    return router.eval()


def router_diagnostics(router, summaries):
    g = router.gate().detach().cpu().numpy()
    with torch.no_grad():
        inv, clu = router.raw_branches(torch.as_tensor(summaries, dtype=torch.float32))
    return {
        "gate_mean": float(g.mean()), "gate_min": float(g.min()), "gate_max": float(g.max()),
        "gate_effective_inv_dims": float(g.sum()), "gate_hard_inv_dims": int((g >= .5).sum()),
        "gate_saturation": float(((g < .05) | (g > .95)).mean()),
        "summary_inv_norm": float(inv.norm(dim=1).mean()),
        "summary_clu_norm": float(clu.norm(dim=1).mean()),
        "summary_inv_zero_fraction": float((inv.norm(dim=1) <= 1e-8).float().mean()),
        "summary_clu_zero_fraction": float((clu.norm(dim=1) <= 1e-8).float().mean()),
    }


## 4. Local training and client summaries
For `full`, retain the source's server-routed normalized mean. For `cluster_only`, normalize each sample's cluster branch **before** averaging, then normalize the client mean. All participating clients use the same broadcast router for this extraction; the updated router is broadcast next round. The invariant branch still supplies CALD prototypes.

CAE warm-up, if selected, redistributes five of the existing 100 AE local-update rounds before classifier round 1, with no prototypes. The final five classifier rounds still train normally but do not update the CAE. The warm-up uses those same five scheduled participant sets, preserving each client's AE example budget, including partial participation. Extra communication/summary cost is recorded, not hidden.


In [ ]:
def train_one_client(
    client_id, classifier_state, ae_state, loader, cfg, in_channels,
    image_size, num_classes, prev_cluster, prototypes, device, round_idx,
    summary_loader=None, summary_router=None, train_ae=True,
):
    classifier = SimpleCNN(in_channels, num_classes).to(device)
    classifier.load_state_dict(classifier_state)
    classifier.train()
    autoencoder = None
    if cfg.uses_latentfed:
        autoencoder = make_autoencoder(in_channels, image_size, cfg).to(device)
        if ae_state is not None:
            autoencoder.load_state_dict(ae_state)
        autoencoder.train()
    optimizer = torch.optim.Adam(classifier.parameters(), lr=cfg.classifier_lr, weight_decay=cfg.weight_decay)
    ae_optimizer = None
    if autoencoder is not None and train_ae:
        groups = [{"params": [p for name, p in autoencoder.named_parameters() if not name.startswith("router.")], "weight_decay": cfg.weight_decay}]
        routing_params = [p for p in autoencoder.router.parameters() if p.requires_grad]
        if routing_params:
            groups.append({"params": routing_params, "weight_decay": 0.0})
        ae_optimizer = torch.optim.Adam(groups, lr=cfg.autoencoder_lr)
    reference = {name: p.detach().clone() for name, p in classifier.named_parameters()} if cfg.uses_fedprox else None
    totals = {key: 0.0 for key in ("loss_ce", "loss_prox", "loss_rec", "loss_structure", "loss_cald_inv", "loss_cald_clu", "loss_cald_margin", "loss_route_soft", "loss_route_budget")}
    total_loss, total_correct, total_seen = 0.0, 0, 0
    for _ in range(cfg.local_epochs):
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            if ae_optimizer is not None:
                ae_optimizer.zero_grad(set_to_none=True)
            logits = classifier(x)
            losses = {key: x.new_zeros(()) for key in totals}
            losses["loss_ce"] = F.cross_entropy(logits, y)
            if reference is not None:
                losses["loss_prox"] = .5 * cfg.fedprox_mu * sum((p - reference[name]).square().sum() for name, p in classifier.named_parameters())
            if autoencoder is not None and train_ae:
                recon, z = autoencoder(x)
                losses["loss_rec"] = cfg.lambda_rec * F.mse_loss(recon, x)
                losses["loss_structure"] = cfg.lambda_str * class_structured_loss(z, y, cfg.margin_cls)
                _, cald_terms = cald_loss(z, prev_cluster, prototypes, round_idx, cfg, autoencoder.router)
                losses.update({"loss_" + key: value for key, value in cald_terms.items()})
            loss = sum(losses.values())
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Nonfinite training loss: client={client_id}, round={round_idx}")
            loss.backward()
            optimizer.step()
            if ae_optimizer is not None:
                ae_optimizer.step()
            size = y.numel()
            total_seen += size
            total_correct += int((logits.argmax(1) == y).sum().item())
            total_loss += float(loss.detach()) * size
            for key, value in losses.items():
                totals[key] += float(value.detach()) * size
    if total_seen == 0:
        raise ValueError(f"Empty training loader for client {client_id}")
    summary = compute_client_summary(autoencoder, summary_loader if summary_loader is not None else loader, device, summary_router) if autoencoder is not None else None
    return {"client_id": client_id, "classifier_state": state_dict_to_cpu(classifier.state_dict()),
            "ae_state": state_dict_to_cpu(autoencoder.state_dict()) if autoencoder is not None else None,
            "summary": summary, "num_train": len(loader.dataset), "num_seen": total_seen,
            "train_loss": total_loss / total_seen, "train_acc": total_correct / total_seen,
            **{key: value / total_seen for key, value in totals.items()}}




@torch.no_grad()
def compute_client_summary(autoencoder, loader, device, router=None):
    autoencoder.eval()
    # One shared broadcast mask, independent of the local router's update.
    router = copy.deepcopy(router if router is not None else autoencoder.router).to(device).eval()
    totals = {}
    count = 0
    for x, _ in loader:
        z = autoencoder.encode(x.to(device, non_blocking=True))
        inv, clu = router(z)
        for key, value in (("raw", z), ("inv", inv), ("clu", clu)):
            total = value.double().sum(0).cpu().numpy()
            totals[key] = totals.get(key, 0) + total
        count += len(x)
    if count == 0:
        raise ValueError("Empty summary loader")
    return {key: l2_normalize_np((value / count)[None])[0] for key, value in totals.items()}


def warmup_one_client(ae_state, loader, cfg, channels, size, device):
    ae = make_autoencoder(channels, size, cfg).to(device)
    ae.load_state_dict(ae_state)
    ae.train()
    # Reconstruction + class structure only; no CALD/router or classifier updates.
    optimizer = torch.optim.Adam([p for name, p in ae.named_parameters()
        if p.requires_grad and not name.startswith("router.")], lr=cfg.autoencoder_lr, weight_decay=cfg.weight_decay)
    count, total = 0, 0.0
    for _ in range(cfg.local_epochs):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad(set_to_none=True)
            recon, z = ae(x)
            loss = cfg.lambda_rec * F.mse_loss(recon, x) + cfg.lambda_str * class_structured_loss(z, y, cfg.margin_cls)
            if not torch.isfinite(loss):
                raise FloatingPointError("Nonfinite warm-up loss")
            loss.backward()
            optimizer.step()
            count += len(y)
            total += float(loss.detach()) * len(y)
    return dict(ae_state=state_dict_to_cpu(ae.state_dict()), num_train=len(loader.dataset),
                num_seen=count, loss=total / count)


## 5. Geometry, alignment, routing and prototype updates
Raw HDBSCAN labels are never overwritten. Intrinsic metrics and production prototypes use original inliers. Routing is a separate model-selection operation: retain the preceding **valid** cluster if present, otherwise choose the nearest current cosine centroid. An all-noise round retains prior models/prototypes (or model 0 before any cluster exists).

Hungarian matching includes unmatched dummy columns and rejects cosine distance > 1. New IDs increase monotonically. Prototype EMA applies only to matched identities; summary EMA is optional. Temporal ARI uses consecutive observed inlier overlap and is invariant to label permutations; ID-switch fraction separately measures alignment behavior.

An auxiliary observed-overlap ARI including noise and the noise-transition fraction expose changes hidden by inlier-only temporal coverage. These are diagnostics, not substitutes for the primary metric.


In [ ]:
def route_summaries(summary_matrix, router, cfg):
    with torch.no_grad():
        inv, clu = router(torch.as_tensor(summary_matrix, dtype=torch.float32))
    inv, clu = inv.numpy().astype(np.float64), clu.numpy().astype(np.float64)
    full = np.concatenate([inv, clu], axis=1) / np.sqrt(2.0)
    if cfg.cald_mode in {"off", "legacy"}:
        full = summary_matrix.copy()
    return full, inv, clu


def make_geometry(summaries, router, cfg, client_ids, cache, round_idx):
    raw = np.stack([s["raw"] for s in summaries])
    if cfg.dp_noise_std:
        raw = l2_normalize_np(raw + np.random.default_rng(role_seed(cfg.seed, "summary-noise", round_idx)).normal(0, cfg.dp_noise_std, raw.shape))
    if cfg.cluster_geometry == "full":
        geometry, inv, clu = route_summaries(raw, router, cfg)
    else:
        inv, clu = (np.stack([s[key] for s in summaries]) for key in ("inv", "clu"))
        geometry = clu.copy()
        if cfg.dp_noise_std:
            geometry = l2_normalize_np(geometry + np.random.default_rng(role_seed(cfg.seed, "cluster-noise", round_idx)).normal(0, cfg.dp_noise_std, geometry.shape))
    for j, client in enumerate(client_ids):
        if cfg.summary_ema_beta and client in cache:
            geometry[j] = unit_ema(cache[client], geometry[j], cfg.summary_ema_beta)
        cache[client] = geometry[j].copy()
    if not all(np.isfinite(x).all() for x in (geometry, inv, clu)):
        raise FloatingPointError("Nonfinite latent summaries")
    return geometry, inv, clu, raw


def cluster_summaries(geometry, cfg):
    if len(geometry) < max(2, cfg.cluster_min_size):
        return np.full(len(geometry), -1, dtype=int)
    if hdbscan is None:
        raise RuntimeError("HDBSCAN is required; no substitute backend")
    return hdbscan.HDBSCAN(min_cluster_size=cfg.cluster_min_size,
        min_samples=cfg.cluster_min_samples, metric="euclidean", prediction_data=False,
        core_dist_n_jobs=1, approx_min_span_tree=True).fit_predict(geometry).astype(int)


def normalized_centroid(rows):
    rows = np.asarray(rows)
    if not len(rows):
        return None
    valid = rows[np.linalg.norm(rows, axis=1) > 1e-8]
    if not len(valid):
        return None
    value = valid.mean(0)
    return l2_normalize_np(value[None])[0] if np.linalg.norm(value) > 1e-8 else None


def unit_ema(previous, current, beta):
    value = beta * previous + (1 - beta) * current
    # Opposite unit vectors can cancel; retain current rather than fabricate a direction.
    return l2_normalize_np(value[None])[0] if np.linalg.norm(value) > 1e-8 else current.copy()


def align_clusters(raw_labels, geometry, previous_prototypes, next_id, cfg):
    raw_ids = sorted(int(k) for k in np.unique(raw_labels) if k >= 0)
    centroids = {k: normalized_centroid(geometry[raw_labels == k]) for k in raw_ids}
    old_ids = sorted(k for k, p in previous_prototypes.items() if p is not None and np.linalg.norm(p) > 1e-8)
    aligned = np.full(len(raw_labels), -1, dtype=int)
    remap, matched = {}, set()
    valid_ids = [k for k in raw_ids if centroids[k] is not None]
    if valid_ids and old_ids:
        cost = 1 - cosine_similarity(np.stack([centroids[k] for k in valid_ids]),
                                     np.stack([previous_prototypes[k] for k in old_ids]))
        threshold = cfg.alignment_max_cosine_distance
        if cfg.cluster_alignment == "hungarian":
            padded = np.concatenate([np.where(cost <= threshold, cost, 1e6),
                np.full((len(valid_ids), len(valid_ids)), threshold + 1e-6)], axis=1)
            rows, cols = linear_sum_assignment(padded)
            pairs = [(i, j) for i, j in zip(rows, cols) if j < len(old_ids) and cost[i, j] <= threshold]
        else:
            pairs, used = [], set()
            for i in range(len(valid_ids)):
                available = [j for j in range(len(old_ids)) if j not in used]
                if available:
                    j = min(available, key=lambda j: cost[i, j])
                    if cost[i, j] <= threshold:
                        pairs.append((i, j)); used.add(j)
        for i, j in pairs:
            remap[valid_ids[i]] = old_ids[j]
            matched.add(old_ids[j])
    for k in raw_ids:
        if k not in remap:
            remap[k] = next_id
            next_id += 1
        aligned[raw_labels == k] = remap[k]
    current = {remap[k]: centroids[k] for k in raw_ids if centroids[k] is not None}
    return aligned, current, matched, next_id


def route_outliers(aligned, geometry, current_centroids, previous_valid, previous_routes, existing_models):
    routed = aligned.copy()
    for i in np.flatnonzero(aligned < 0):
        if previous_valid[i] in current_centroids:
            routed[i] = previous_valid[i]
        elif current_centroids:
            ids = sorted(current_centroids)
            sims = cosine_similarity(geometry[i:i+1], np.stack([current_centroids[k] for k in ids]))[0]
            routed[i] = ids[int(np.argmax(sims))]
        else:
            # There is no current valid cluster: preserve an available model only.
            routed[i] = previous_routes[i] if previous_routes[i] in existing_models else min(existing_models)
    assert (routed >= 0).all()
    return routed


def update_prototypes(geometry, inv, clu, aligned, routed, raw_labels, previous, matched, cfg):
    eligible = raw_labels >= 0 if cfg.exclude_noise_from_prototypes else np.ones(len(raw_labels), bool)
    labels = aligned if cfg.exclude_noise_from_prototypes else routed
    if not eligible.any():
        return copy.deepcopy(previous)
    output = {"inv": None, "cluster": {}, "full_cluster": {}}
    current_inv = normalized_centroid(inv[eligible])
    old_inv = previous.get("inv")
    output["inv"] = (unit_ema(old_inv, current_inv, cfg.prototype_ema_beta)
                     if current_inv is not None and old_inv is not None else current_inv)
    for k in sorted(int(k) for k in np.unique(labels[eligible]) if k >= 0):
        for key, features in (("cluster", clu), ("full_cluster", geometry)):
            value = normalized_centroid(features[eligible & (labels == k)])
            if value is None:
                continue
            old = previous.get(key, {}).get(k)
            output[key][k] = unit_ema(old, value, cfg.prototype_ema_beta) if k in matched and old is not None else value
    return output


## 6. Evaluation and exact seed aggregation
Accuracy uses routed model groups. Clustering K counts raw non-noise groups, while `K_models` counts model groups. Silhouette, CH, DB and manual SSE use **only** HDBSCAN inliers. Fewer than two clusters gives NaN; Silhouette/CH/DB are also undefined with one cluster per observation. SSE is not normalized by feature dimension; compare it alongside SSE/inlier, geometry and K. CH accepts every strictly positive within-cluster SSE; exactly zero SSE retains NaN. Table B uses scientific notation for SSE and SSE/inlier to preserve small means and SDs; numeric exports retain full precision. Saved clustering diagnostics and seed scores are refreshed from stored geometry and raw labels when a checkpoint is loaded or exported.

A seed score requires all rounds 1–100 and averages exactly rounds 91–100. Undefined diagnostic rounds remain NaN; finite-round coverage is recorded. A publication mean/SD is only printed when all five seed scores are finite. Missing seed runs block the table; runs are never silently dropped.


In [ ]:
@torch.no_grad()
def evaluate_model(model: nn.Module, loader: DataLoader, device: torch.device) -> tuple[int, int, float]:
    model.eval()
    correct = 0
    total = 0
    loss_sum = 0.0
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = model(x)
        loss = F.cross_entropy(logits, y, reduction="sum")
        pred = logits.argmax(dim=1)
        correct += int((pred == y).sum().item())
        total += int(y.numel())
        loss_sum += float(loss.item())
    return correct, total, loss_sum / max(1, total)






def evaluate_table_iii_metrics(
    cluster_model_states: dict[int, dict[str, torch.Tensor]],
    client_assignments: np.ndarray,
    client_test_loaders: list[DataLoader],
    cfg: ExperimentConfig,
    in_channels: int,
    num_classes: int,
    device: torch.device,
    return_client_rows: bool = False,
    personalized_states=None,
):
    client_rows = []
    cluster_totals: dict[int, dict[str, float]] = {}

    for client_id, loader in enumerate(client_test_loaders):
        cluster_id = int(client_assignments[client_id])
        if cluster_id not in cluster_model_states:
            raise ValueError("Client routed to missing model")
        model = SimpleCNN(in_channels, num_classes).to(device)
        model.load_state_dict(personalized_states[client_id] if personalized_states is not None else cluster_model_states[cluster_id])
        correct, total, loss = evaluate_model(model, loader, device)
        if total == 0:
            raise ValueError(f"Client {client_id} has no evaluation examples; repair the partition")
        acc = correct / total
        client_rows.append({"client_id": client_id, "cluster_id": cluster_id, "correct": correct, "total": total, "acc": acc, "loss": loss, "evaluation_split": cfg.evaluation_split})
        cluster_totals.setdefault(cluster_id, {"correct": 0, "total": 0})
        cluster_totals[cluster_id]["correct"] += correct
        cluster_totals[cluster_id]["total"] += total
        del model

    if device.type == "cuda":
        torch.cuda.empty_cache()

    client_df = pd.DataFrame(client_rows)
    weighted_acc = client_df["correct"].sum() / max(1, client_df["total"].sum())
    macro_acc = float(client_df["acc"].mean())
    cluster_accs = [v["correct"] / max(1, v["total"]) for v in cluster_totals.values()]

    unique_clusters = np.unique(client_assignments)
    result = {
        "K": int(len(unique_clusters)),
        "Worst-client-Acc": float(client_df["acc"].min()),
        "P10-client-Acc": float(client_df["acc"].quantile(0.1)),
        "W-Acc": float(weighted_acc),
        "M-Acc": float(macro_acc),
        "MC": float(np.mean(cluster_accs)) if cluster_accs else float("nan"),
        "BC": float(np.max(cluster_accs)) if cluster_accs else float("nan"),
        "WC": float(np.min(cluster_accs)) if cluster_accs else float("nan"),
    }
    return (result, client_rows) if return_client_rows else result



def clustering_metrics(features, labels):
    labels = np.asarray(labels)
    mask = labels >= 0
    k, ni = len(np.unique(labels[mask])), int(mask.sum())
    result = dict(K=k, n_inliers=ni, n_outliers=int((~mask).sum()),
                  noise_fraction=float((~mask).mean()) if len(mask) else float("nan"))
    result.update({key: float("nan") for key in ("Silhouette", "Calinski-Harabasz", "Davies-Bouldin", "SSE", "SSE/Inlier")})
    if features is None:
        return result
    x = np.asarray(features, dtype=np.float64)
    if x.ndim != 2 or len(x) != len(labels) or not np.isfinite(x).all():
        raise ValueError("Invalid clustering metric inputs")
    x, y = x[mask], labels[mask]
    if k < 2:
        return result
    sse = sum(float(((x[y == c] - x[y == c].mean(0)) ** 2).sum()) for c in np.unique(y))
    result.update(SSE=sse, **{"SSE/Inlier": sse / ni})
    # CH is scale invariant: any strictly positive within-cluster SSE is valid.
    # Exactly zero SSE (undefined/unbounded CH) retains NaN, as do k == ni cases.
    if k < ni and sse > 0.0:
        result["Calinski-Harabasz"] = float(calinski_harabasz_score(x, y))
    if k < ni and not np.all(np.linalg.norm(x - x[:1], axis=1) <= 1e-12):
        result["Silhouette"] = float(silhouette_score(x, y))
        centers = np.stack([x[y == c].mean(0) for c in np.unique(y)])
        if np.min(np.linalg.norm(centers[:, None] - centers[None, :], axis=-1) + np.eye(k) * 1e9) > 1e-12:
            result["Davies-Bouldin"] = float(davies_bouldin_score(x, y))
    return result


def temporal_metrics(previous, current, selected, previous_selected):
    overlap = np.intersect1d(selected, previous_selected).astype(int)
    ids = overlap[(previous[overlap] >= 0) & (current[overlap] >= 0)]
    valid = len(ids) >= 2 and min(len(np.unique(previous[ids])), len(np.unique(current[ids]))) >= 2
    return {"Temporal ARI": float(adjusted_rand_score(previous[ids], current[ids])) if valid else float("nan"),
            "Temporal ARI with noise": float(adjusted_rand_score(previous[overlap], current[overlap])) if len(overlap) >= 2 else float("nan"),
            "noise_transition_fraction": float(np.mean((previous[overlap] < 0) != (current[overlap] < 0))) if len(overlap) else float("nan"),
            "temporal_observed_overlap": int(len(overlap)), "temporal_inlier_overlap": int(len(ids)),
            "cluster_id_switch_fraction": float(np.mean(previous[ids] != current[ids])) if len(ids) else float("nan")}


CLASS_METRICS = ["W-Acc", "M-Acc", "MC", "BC", "WC"]
CLUSTER_METRICS = ["Silhouette", "Calinski-Harabasz", "Davies-Bouldin", "SSE", "SSE/Inlier",
                   "K", "noise_fraction", "Temporal ARI"]
DIAGNOSTIC_METRICS = ["n_inliers", "n_outliers", "K_models", "cluster_id_switch_fraction",
                      "temporal_inlier_overlap", "Temporal ARI with noise", "noise_transition_fraction", "mdfe_rho_min", "mdfe_rho_mean", "mdfe_rho_max",
                      "mdfe_rho_negative_fraction"]


def seed_summary(history, cfg):
    frame = pd.DataFrame(history)
    if len(frame) != 100 or frame["round"].tolist() != list(range(1, 101)):
        raise ValueError("A seed score requires exactly rounds 1–100, without duplicates or omissions")
    tail = frame.loc[frame["round"].between(91, 100)]
    out = dict(Dataset=cfg.dataset, Distribution="IID" if cfg.alpha is None else f"alpha={cfg.alpha}",
               Method=cfg.method, Seed=cfg.seed, Evaluation=cfg.evaluation_split, config_id=cfg.config_id,
               window_start=91, window_end=100)
    for metric in CLASS_METRICS + CLUSTER_METRICS + DIAGNOSTIC_METRICS:
        values = tail[metric].to_numpy(dtype=float) if metric in tail else np.full(10, np.nan)
        if np.isinf(values).any():
            raise ValueError(f"Infinite metric {metric}")
        count = int(np.isfinite(values).sum())
        if metric in CLASS_METRICS and count != 10:
            raise ValueError(f"Missing classification observations: {metric}")
        out[metric] = float(np.nanmean(values)) if count else float("nan")
        out[metric + "_valid_rounds"] = count
    out["multicluster_fraction"] = float((tail["K"] >= 2).mean())
    out["classifier_examples"] = int(frame["train_examples"].sum())
    out["ae_examples_normal"] = int(frame["ae_examples"].sum())
    return out


def publication_tables(seed_rows):
    columns_a = ["Dataset", "Distribution", "Method"] + CLASS_METRICS
    labels_b = ["Silhouette ↑", "Calinski-Harabasz ↑", "Davies-Bouldin ↓", "SSE ↓", "SSE/Inlier ↓", "K", "Noise Fraction", "Temporal ARI ↑"]
    if not seed_rows:
        return pd.DataFrame(columns=columns_a), pd.DataFrame(columns=columns_a[:3] + labels_b), pd.DataFrame()
    frame = pd.DataFrame(seed_rows)
    if not (frame["Evaluation"] == "test").all():
        raise ValueError("Final publication tables accept only frozen test runs")
    a, b, coverage = [], [], []
    for keys, group in frame.groupby(columns_a[:3], sort=True):
        if sorted(group.Seed.tolist()) != sorted(FINAL_SEEDS):
            raise ValueError(f"Incomplete or duplicated official seeds for {keys}")
        if not ((group.window_start == 91) & (group.window_end == 100)).all():
            raise ValueError("Wrong final-round window")
        base = dict(zip(columns_a[:3], keys))
        row_a, row_b = base.copy(), base.copy()
        for metric, label in list(zip(CLASS_METRICS, CLASS_METRICS)) + list(zip(CLUSTER_METRICS, labels_b)):
            values = group[metric].to_numpy(float)
            finite = np.isfinite(values)
            coverage.append({**base, "Metric": metric, "Valid seeds": int(finite.sum()),
                "Valid rounds per seed (official order)": [int(group.loc[group.Seed == seed, metric + "_valid_rounds"].iloc[0]) for seed in FINAL_SEEDS]})
            if finite.all():
                scale = 100 if metric in CLASS_METRICS else 1
                # Scientific notation preserves small positive SSE means and sample SDs.
                precision = ".3e" if metric in {"SSE", "SSE/Inlier"} else ".2f"
                text = f"{scale * values.mean():{precision}} ± {scale * values.std(ddof=1):{precision}}"
            else:
                text = f"NaN ({finite.sum()}/5 valid seeds)"
            (row_a if metric in CLASS_METRICS else row_b)[label] = text
        a.append(row_a); b.append(row_b)
    return pd.DataFrame(a, columns=columns_a), pd.DataFrame(b, columns=columns_a[:3] + labels_b), pd.DataFrame(coverage)

CLUSTER_METRICS_VERSION = "inlier-metrics-v2-positive-sse-ch"


def refresh_saved_clustering_metrics(suite):
    """Refresh diagnostics from saved geometry/labels without retraining or changing routes."""
    runs = list(suite["runs"].values())
    if suite.get("active") is not None:
        runs.append(suite["active"])
    for run in runs:
        if run.get("cluster_metrics_version") == CLUSTER_METRICS_VERSION:
            continue
        for row in run["history"]:
            if "clustering_geometry" in row and "raw_labels" in row:
                row.update(clustering_metrics(row["clustering_geometry"], row["raw_labels"]))
        if run.get("seed_score") is not None:
            run["seed_score"] = seed_summary(run["history"], ExperimentConfig(**run["config"]))
        run["cluster_metrics_version"] = CLUSTER_METRICS_VERSION



## 7. Federated loop and one restart bundle
The bundle contains completed histories, development decisions, frozen configuration, and the active model/RNG/prototype/EMA state. Periodic checkpoints replace the same file atomically. Explicit resume verifies config, data, initialization and package identity. No external uploads. Each completed experiment remains in `SUITE["runs"]`, including per-round client counts, labels and latent diagnostics.


In [ ]:
def capture_rng_state():
    return dict(python=random.getstate(), numpy=np.random.get_state(), torch=torch.get_rng_state(),
                cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None)


def restore_rng_state(state):
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    if state["cuda"] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(state["cuda"])


def fresh_suite():
    return dict(version=METHOD_IMPLEMENTATION_VERSION, runs={}, active=None, decisions=[],
                frozen=None, development_complete=False)


def save_bundle(suite):
    if not ENABLE_CHECKPOINT:
        return
    CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
    temporary = CHECKPOINT_PATH.with_suffix(".tmp")
    torch.save(suite, temporary)
    os.replace(temporary, CHECKPOINT_PATH)


# Re-running configuration/definition cells must not discard completed seed results.
if "SUITE" not in globals():
    if RESUME_BUNDLE:
        SUITE = safe_torch_load(RESUME_PATH)
    else:
        if ENABLE_CHECKPOINT and CHECKPOINT_PATH.exists():
            raise FileExistsError("Saved results exist. Rerun the configuration cell to auto-resume, or set RESUME_FROM to your checkpoint.")
        SUITE = fresh_suite()
if SUITE.get("version") != METHOD_IMPLEMENTATION_VERSION:
    raise ValueError("Incompatible revision bundle")
refresh_saved_clustering_metrics(SUITE)


def config_digest(payload):
    return hashlib.sha256(json.dumps(payload, sort_keys=True, allow_nan=False).encode()).hexdigest()


def validate_run_phase(cfg, phase, suite):
    cfg.validate()
    if phase == "smoke":
        if not cfg.allow_custom_protocol or cfg.seed not in DEV_SEEDS:
            raise ValueError("Smoke requires a separate development seed and custom short budget")
        return
    if cfg.rounds != 100 or cfg.allow_custom_protocol:
        raise ValueError("Research runs require 100 rounds and the unchanged classifier protocol")
    if phase == "development":
        if cfg.seed not in DEV_SEEDS or cfg.evaluation_split != "validation":
            raise ValueError("Development may use only DEV_SEEDS and validation")
        if suite.get("frozen") is not None:
            raise ValueError("Configuration already frozen: development is closed")
    elif phase == "final":
        frozen = suite.get("frozen")
        if cfg.seed not in FINAL_SEEDS or cfg.evaluation_split != "test" or frozen is None:
            raise ValueError("Final test runs require official seeds and a frozen configuration")
        payload = {k: v for k, v in frozen.items() if k != "digest"}
        if config_digest(payload) != frozen["digest"]:
            raise ValueError("Frozen configuration was altered")
        expected = replace(ExperimentConfig(**frozen["config"]), dataset=cfg.dataset,
                           alpha=cfg.alpha, seed=cfg.seed, evaluation_split="test")
        if cfg.config_id != expected.config_id or cfg.dataset not in frozen["datasets"] or cfg.alpha not in frozen["alphas"]:
            raise ValueError("Final run differs from frozen settings")
    else:
        raise ValueError("Unknown experiment phase")


def aggregate_client_models(client_results, assignments):
    groups = {}
    for result in client_results:
        groups.setdefault(int(assignments[result["client_id"]]), []).append(result)
    return {k: weighted_average_states([r["classifier_state"] for r in rows], [r["num_train"] for r in rows])
            for k, rows in groups.items()}


def run_experiment(cfg, phase="development", suite=None, dataset_bundle=None, stop_after=None):
    suite = SUITE if suite is None else suite
    validate_run_phase(cfg, phase, suite)
    key = phase + ":" + cfg.config_id
    if key in suite["runs"]:
        return suite["runs"][key]
    if phase != "smoke" and dataset_bundle is not None:
        raise ValueError("Synthetic/custom datasets are permitted only for smoke tests")
    set_seed(cfg.seed)
    device = get_device()
    bundle = dataset_bundle if dataset_bundle is not None else get_dataset_bundle(cfg.dataset, cfg.data_root)
    _, _, _, _, channels, size, classes = bundle
    fair = FairRunData(cfg, bundle)
    active = suite.get("active")
    if active is not None:
        if active["key"] != key:
            raise ValueError("Resume the active experiment before starting another")
        for identity in ("partition_hash", "initial_state_hash", "schedule_hash", "packages", "device"):
            if active["manifest"][identity] != fair.manifest[identity]:
                raise ValueError(f"Resume identity mismatch: {identity}")
        state = copy.deepcopy(active)
        restore_rng_state(state["rng"])
    else:
        state = dict(key=key, phase=phase, config=asdict(cfg), manifest=fair.manifest,
            round=0, warmup_done=0, warmup_history=[], history=[],
            cluster_models={0: copy.deepcopy(fair.base_state)}, assignments=np.zeros(cfg.num_clients, dtype=int),
            previous_valid=np.full(cfg.num_clients, -1, dtype=int), previous_selected=[],
            prototypes={}, next_cluster_id=1, summary_cache={},
            ae_state=isolated_model_state(lambda: make_autoencoder(channels, size, cfg), cfg, "autoencoder-initialization") if cfg.uses_latentfed else None)
    def checkpoint():
        state["rng"] = capture_rng_state()
        suite["active"] = copy.deepcopy(state)
        if phase != "smoke":
            save_bundle(suite)
    started = time.monotonic()
    warmup = cfg.ae_pretrain_rounds if cfg.uses_latentfed else 0
    for w in range(state["warmup_done"], warmup):
        # Move the last AE rounds to initialization: same clients and example counts.
        budget_round = cfg.rounds - warmup + w + 1
        clients = fair.schedule[budget_round - 1]
        fair.set_round(-w - 1)
        rows = []
        for client in clients:
            set_seed(role_seed(cfg.seed, "ae-warmup", w, client))
            rows.append(warmup_one_client(state["ae_state"], fair.train_loaders[client], cfg, channels, size, device))
        state["ae_state"] = weighted_average_states([r["ae_state"] for r in rows], [r["num_train"] for r in rows])
        state["warmup_history"].append(dict(warmup_round=w + 1, moved_from_round=budget_round,
            selected_clients=clients, ae_examples=sum(r["num_seen"] for r in rows),
            reconstruction_structure_loss=float(np.average([r["loss"] for r in rows], weights=[r["num_seen"] for r in rows]))))
        state["warmup_done"] = w + 1
        checkpoint()
    for r in tqdm(range(state["round"] + 1, cfg.rounds + 1), desc=cfg.run_name, disable=phase == "smoke"):
        fair.set_round(r)
        clients = fair.schedule[r - 1]
        old_routes = state["assignments"].copy()
        old_valid = state["previous_valid"].copy()
        train_ae = cfg.uses_latentfed and r <= cfg.rounds - warmup
        broadcast_router = server_router(state["ae_state"], cfg) if cfg.uses_latentfed else None
        rows = []
        for client in clients:
            set_seed(role_seed(cfg.seed, "local-stochastic", r, client))
            rows.append(train_one_client(client, state["cluster_models"][int(old_routes[client])],
                state["ae_state"], fair.train_loaders[client], cfg, channels, size, classes,
                int(old_routes[client]), state["prototypes"], device, r,
                summary_loader=fair.assignment_loaders[client], summary_router=broadcast_router, train_ae=train_ae))
        diagnostics = {}
        geometry = None
        raw = np.full(len(clients), -1, dtype=int)
        aligned = raw.copy()
        if cfg.uses_latentfed:
            state["ae_state"] = weighted_average_states([x["ae_state"] for x in rows], [x["num_train"] for x in rows])
            router = server_router(state["ae_state"], cfg)
            geometry, inv, clu, summaries = make_geometry([x["summary"] for x in rows], router, cfg, clients, state["summary_cache"], r)
            raw = cluster_summaries(geometry, cfg)
            aligned, centers, matched, state["next_cluster_id"] = align_clusters(raw, geometry,
                state["prototypes"].get("full_cluster", {}), state["next_cluster_id"], cfg)
            routed = route_outliers(aligned, geometry, centers, old_valid[clients], old_routes[clients], state["cluster_models"])
            state["assignments"][clients] = routed
            state["prototypes"] = update_prototypes(geometry, inv, clu, aligned, routed, raw,
                state["prototypes"], matched, cfg)
            diagnostics.update(router_diagnostics(router, summaries))
            diagnostics.update(mdfe_parameter_diagnostics(state["ae_state"], cfg))
            diagnostics.update(prototype_eligible_clients=int((raw >= 0).sum()) if cfg.exclude_noise_from_prototypes else len(raw),
                geometry_dimension=geometry.shape[1], prototype_dimension=clu.shape[1],
                matched_clusters=len(matched), raw_labels=raw.tolist(), aligned_labels=aligned.tolist(),
                routed_labels=routed.tolist(), clustering_geometry=geometry.tolist())
        else:
            state["assignments"].fill(0)
        current_valid = old_valid.copy()  # retain last observed validity for nonparticipants
        current_valid[clients] = aligned
        temporal = temporal_metrics(old_valid, current_valid, clients, state["previous_selected"])
        state["cluster_models"].update(aggregate_client_models(rows, state["assignments"]))
        # Retain models needed by nonparticipants; discard unreferenced obsolete states.
        state["cluster_models"] = {k: v for k, v in state["cluster_models"].items() if k in set(state["assignments"].tolist())}
        metrics, client_metrics = evaluate_table_iii_metrics(state["cluster_models"], state["assignments"],
            fair.evaluation_loaders, cfg, channels, classes, device, return_client_rows=True)
        metrics["K_models"] = metrics.pop("K")
        metrics.update(clustering_metrics(geometry, raw))
        weights = [x["num_seen"] for x in rows]
        record = dict(round=r, selected_clients=clients, evaluation_split=cfg.evaluation_split,
            train_examples=int(sum(weights)), ae_examples=int(sum(weights)) if train_ae else 0,
            cald_kappa=cald_curriculum(r, cfg), client_metrics=client_metrics,
            assignments=state["assignments"].tolist(),
            train_loss=float(np.average([x["train_loss"] for x in rows], weights=weights)),
            train_acc=float(np.average([x["train_acc"] for x in rows], weights=weights)),
            **metrics, **diagnostics, **temporal)
        for field in rows[0]:
            if field.startswith("loss_"):
                record[field] = float(np.average([x[field] for x in rows], weights=weights))
        if not record["WC"] - 1e-10 <= record["W-Acc"] <= record["BC"] + 1e-10:
            raise AssertionError("Invalid weighted classification metric")
        state["history"].append(record)
        state.update(round=r, previous_valid=current_valid, previous_selected=clients)
        if r % cfg.checkpoint_every == 0 or r == cfg.rounds or r == stop_after:
            checkpoint()
        if phase != "smoke" and (r == 1 or r % 10 == 0):
            print(f"round {r}: W-Acc={record['W-Acc']:.4f}, K={record['K']}, noise={record['noise_fraction']:.2f}")
        if stop_after is not None and r >= stop_after and r < cfg.rounds:
            return dict(interrupted=True, history=copy.deepcopy(state["history"]))
    expected = sum(len(fair.train_parts[c]) * cfg.local_epochs for ids in fair.schedule for c in ids)
    ae_examples = sum(x["ae_examples"] for x in state["history"]) + sum(x["ae_examples"] for x in state["warmup_history"])
    classifier_examples = sum(x["train_examples"] for x in state["history"])
    if classifier_examples != expected or (cfg.uses_latentfed and ae_examples != expected):
        raise AssertionError("Local example budget changed")
    result = dict(config=asdict(cfg), phase=phase, manifest=fair.manifest, history=state["history"],
        warmup_history=state["warmup_history"], ae_examples=ae_examples, classifier_examples=classifier_examples,
        elapsed_this_session_seconds=time.monotonic() - started,
        seed_score=seed_summary(state["history"], cfg) if phase != "smoke" else None)
    suite["runs"][key] = result
    suite["active"] = None
    if phase != "smoke":
        save_bundle(suite)
    return result


## 8. Smoke tests — run before any development or final experiment
Synthetic images exercise both input shapes, actual local optimization, clustering, prototype dimensions, partial participation, exact resume, fair CAE budgets, and statistics. The fixture is isolated and cannot enter development/final tables. Tests use no downloads or output files.


In [ ]:
class SyntheticImages(Dataset):
    def __init__(self, count, channels, size, classes, seed):
        generator = torch.Generator().manual_seed(seed)
        self.images = torch.randn(count, channels, size, size, generator=generator)
        self.targets = np.arange(count) % classes
        self.transform = None
    def __len__(self):
        return len(self.targets)
    def __getitem__(self, index):
        return self.images[index], int(self.targets[index])


def synthetic_bundle(channels=3, size=32, classes=3):
    tr = SyntheticImages(48, channels, size, classes, 7)
    te = SyntheticImages(24, channels, size, classes, 19)
    return tr, te, tr.targets, te.targets, channels, size, classes


def run_smoke_tests():
    old_rng = capture_rng_state()
    old_threads = torch.get_num_threads()
    torch.set_num_threads(1)
    passed = []
    try:
        # Native CPU pooling is the independent forward/backward reference.
        # When CUDA exists, exercise the replacement there under strict determinism.
        pool_device = get_device()
        set_seed(7)
        for shape, output_size in (((2, 3, 4, 4), (4, 4)), ((2, 3, 3, 3), (4, 4)),
                                   ((2, 3, 7, 5), (4, 4)), ((2, 3, 2, 3), (4, 4))):
            reference_x = torch.randn(*shape, dtype=torch.float64, requires_grad=True)
            candidate_x = reference_x.detach().to(pool_device).requires_grad_(True)
            reference_y = F.adaptive_avg_pool2d(reference_x, output_size)
            candidate_y = DeterministicAdaptiveAvgPool2d(output_size)(candidate_x)
            upstream = torch.randn_like(reference_y)
            reference_y.backward(upstream)
            candidate_y.backward(upstream.to(pool_device))
            assert torch.allclose(reference_y, candidate_y.detach().cpu(), atol=1e-12, rtol=1e-12)
            assert torch.allclose(reference_x.grad, candidate_x.grad.cpu(), atol=1e-12, rtol=1e-12)
        assert not DeterministicAdaptiveAvgPool2d((4, 4)).state_dict()
        assert torch.are_deterministic_algorithms_enabled()
        assert not torch.is_deterministic_algorithms_warn_only_enabled()
        passed.append(f"Adaptive pooling matches native CPU values/gradients; strict backward on {pool_device}")
        cfg = replace(CFG, seed=7, num_clients=4, participation_ratio=1.0, rounds=3,
                      batch_size=8, num_workers=0, allow_custom_protocol=True,
                      min_train_samples=1, min_validation_samples=1, min_test_samples=1,
                      alpha=None, cald_warmup_rounds=0, cald_ramp_rounds=1,
                      cluster_geometry="cluster_only", cluster_alignment="hungarian",
                      prototype_ema_beta=.5, summary_ema_beta=.5, ae_pretrain_rounds=1)
        for channels, size in ((3, 32), (1, 28)):
            classifier = SimpleCNN(channels, 10 if channels == 3 else 47).to(pool_device)
            classifier_input = torch.randn(4, channels, size, size, device=pool_device)
            F.cross_entropy(classifier(classifier_input), torch.arange(4, device=pool_device)).backward()
            assert all(torch.isfinite(p.grad).all() for p in classifier.parameters() if p.grad is not None)
            del classifier, classifier_input
            for mode in ("fixed_split", "adaptive"):
                c = replace(cfg, cald_mode=mode)
                ae = make_autoencoder(channels, size, c)
                x = torch.randn(4, channels, size, size)
                recon, z = ae(x)
                inv, clu = ae.router(z)
                dim = 64 if mode == "fixed_split" else 128
                assert z.shape == (4, 128) and inv.shape == clu.shape == (4, dim) and recon.shape == x.shape
                if mode == "fixed_split":
                    assert torch.equal(ae.router.raw_branches(z)[0], z[:, :64])
                    assert torch.equal(ae.router.raw_branches(z)[1], z[:, 64:])
                protos = {"inv": inv.detach().mean(0).numpy(), "cluster": {1: clu.detach().mean(0).numpy(), 2: -clu.detach().mean(0).numpy()}}
                loss, _ = cald_loss(z, 1, protos, 2, c, ae.router)
                (loss + F.mse_loss(recon, x)).backward()
                assert all(torch.isfinite(t.grad).all() for t in ae.parameters() if t.grad is not None)
        passed.append("CIFAR/EMNIST shapes; fixed/adaptive CALD dimensions and finite gradients")
        gate = MDFEGate(4)
        gate.configure_rho("positive")
        assert torch.allclose(gate.effective_rho(), torch.ones(4))
        gate.rho_raw.data.fill_(-20)
        d = torch.linspace(0, 5, 20)[:, None]
        weights = torch.sigmoid(-F.softplus(gate.gamma) * torch.sigmoid(d * gate.effective_rho() + gate.zeta))
        assert (weights[1:] <= weights[:-1]).all() and (gate.effective_rho() > 0).all()
        gate.configure_rho("original"); gate.rho_raw.data.fill_(-1)
        assert (gate.effective_rho() < 0).all()
        passed.append("Positive rho initialization and monotone distortion path; original negative control")
        x = np.array([[1., 0], [1., .1], [0, 1], [.1, 1], [100, 100]])
        raw = np.array([0, 0, 1, 1, -1])
        metrics = clustering_metrics(x, raw)
        assert np.isclose(metrics["SSE"], .01) and metrics["n_inliers"] == 4 and metrics["n_outliers"] == 1
        assert np.isclose(metrics["Silhouette"], silhouette_score(x[:4], raw[:4]))
        assert np.isclose(metrics["Calinski-Harabasz"], calinski_harabasz_score(x[:4], raw[:4]))
        assert np.isclose(metrics["Davies-Bouldin"], davies_bouldin_score(x[:4], raw[:4]))
        # Independent CH reference: within SSE=.01, between SSE=2, n=4, k=2.
        compact = np.array([[0., 0.], [.1, 0.], [1., 1.], [1.1, 1.]])
        compact_labels = np.array([0, 0, 1, 1])
        for scale in (1., 1e-6, 1e-14):
            compact_metrics = clustering_metrics(compact * scale, compact_labels)
            assert np.isclose(compact_metrics["Calinski-Harabasz"], 400., rtol=1e-12)
            assert np.isclose(compact_metrics["SSE"], .01 * scale ** 2, rtol=1e-12, atol=0.)
        unit_compact = np.array([[1., 0.], [1., 1e-7], [0., 1.], [1e-7, 1.]])
        unit_compact /= np.linalg.norm(unit_compact, axis=1, keepdims=True)
        assert np.isfinite(clustering_metrics(unit_compact, compact_labels)["Calinski-Harabasz"])
        zero_sse = clustering_metrics(np.array([[0., 0.]] * 2 + [[1., 1.]] * 2), compact_labels)
        assert zero_sse["SSE"] == 0. and np.isnan(zero_sse["Calinski-Harabasz"])
        passed.append("CH scale invariance, tiny positive SSE and exactly zero SSE policy")
        for labels in (np.full(5, -1), np.zeros(5, int)):
            assert np.isnan(clustering_metrics(x, labels)["SSE"])
        assert np.isnan(clustering_metrics(x, np.arange(5))["Silhouette"])
        assert np.isfinite(clustering_metrics(x[:4], np.array([0, 1, 1, 1]))["SSE"])
        labels, centers, matched, next_id = align_clusters(raw, x, {}, 1, cfg)
        routed = route_outliers(labels, x, centers, np.array([-1, -1, -1, -1, 1]), np.zeros(5, int), {0: {}})
        assert routed[-1] == 1 and labels[-1] == -1
        proto = update_prototypes(x, x, x, labels, routed, raw, {}, matched, cfg)
        x2 = x.copy(); x2[-1] = [-900, -900]
        proto2 = update_prototypes(x2, x2, x2, labels, routed, raw, {}, matched, cfg)
        assert np.array_equal(proto["inv"], proto2["inv"])
        for key in ("cluster", "full_cluster"):
            assert all(np.array_equal(proto[key][k], proto2[key][k]) for k in proto[key])
        none_labels, _, _, new_id = align_clusters(np.full(5, -1), x, proto["full_cluster"], next_id, cfg)
        assert new_id == next_id and (none_labels == -1).all()
        assert (route_outliers(none_labels, x, {}, np.full(5, -1), np.zeros(5, int), {0: {}}) == 0).all()
        # Orthogonal-opposite centroid must be born under a fresh identity.
        bad, _, matches, _ = align_clusters(np.array([0, 0]), np.array([[-1., 0], [-1., 0]]), {9: np.array([1., 0])}, 10, cfg)
        assert not matches and (bad == 10).all()
        angle = np.deg2rad(40.)
        features = np.array([[np.cos(angle), np.sin(angle)]] * 2 + [[1., 0.]] * 2)
        previous = {3: np.array([1., 0.]), 4: np.array([0., 1.])}
        hg, _, _, _ = align_clusters(np.array([0, 0, 1, 1]), features, previous, 5, cfg)
        gg, _, _, _ = align_clusters(np.array([0, 0, 1, 1]), features, previous, 5, replace(cfg, cluster_alignment="greedy"))
        def cost(labels):
            return sum(1 - float(np.dot(features[i], previous[int(labels[i])])) for i in (0, 2))
        assert cost(hg) < cost(gg) and hg.tolist() == [4, 4, 3, 3]
        preserved = update_prototypes(x, x, x, none_labels, routed, np.full(5, -1), proto, set(), cfg)
        assert np.array_equal(preserved["inv"], proto["inv"])
        old_proto = {"inv": np.array([1., 0.]), "cluster": {1: np.array([1., 0.])}, "full_cluster": {1: np.array([1., 0.])}}
        ema_proto = update_prototypes(x, x, x, labels, routed, raw, old_proto, {1}, cfg)
        assert np.allclose(ema_proto["cluster"][1], unit_ema(old_proto["cluster"][1], normalized_centroid(x[:2]), .5))
        assert np.allclose(ema_proto["cluster"][2], normalized_centroid(x[2:4]))
        passed.append("Inlier-only Silhouette/CH/DB/manual SSE, outlier exclusion/routing and unmatched cluster IDs")
        old = np.array([1, 1, 2, 2, -1]); new = np.array([7, 7, 9, 9, -1])
        temporal = temporal_metrics(old, new, list(range(5)), list(range(5)))
        assert temporal["Temporal ARI"] == 1 and temporal["temporal_inlier_overlap"] == 4
        assert np.isnan(temporal_metrics(old, new, [0], [0])["Temporal ARI"])
        passed.append("Temporal ARI uses observed inlier overlap and ignores ID permutations")
        bundle = synthetic_bundle()
        first, second = fresh_suite(), fresh_suite()
        full = run_experiment(cfg, "smoke", first, bundle)
        paused = run_experiment(cfg, "smoke", second, bundle, stop_after=1)
        assert paused["interrupted"]
        import io
        buffer = io.BytesIO()
        torch.save(second, buffer)
        buffer.seek(0)
        second = torch.load(buffer, weights_only=False)
        resumed = run_experiment(cfg, "smoke", second, bundle)
        for a, b in zip(full["history"], resumed["history"]):
            for m in CLASS_METRICS + ["train_loss", "SSE", "Temporal ARI"]:
                assert np.allclose(a[m], b[m], equal_nan=True), m
            assert a["assignments"] == b["assignments"]
        assert full["ae_examples"] == full["classifier_examples"]
        baseline_budget = run_experiment(replace(cfg, ae_pretrain_rounds=0), "smoke", fresh_suite(), bundle)
        assert baseline_budget["ae_examples"] == full["ae_examples"]
        em = run_experiment(replace(cfg, dataset="emnist", cald_mode="adaptive", cluster_geometry="full",
             mdfe_rho_mode="original", participation_ratio=.5), "smoke", fresh_suite(), synthetic_bundle(1, 28))
        assert em["ae_examples"] == em["classifier_examples"]
        repeated = run_experiment(cfg, "smoke", fresh_suite(), bundle)
        assert [r["W-Acc"] for r in repeated["history"]] == [r["W-Acc"] for r in full["history"]]
        passed.append("Real synthetic training: 15 synthetic classifier rounds including resumed work, warm-up budget, partial participation and exact resume")
        summaries = []
        for i, seed in enumerate(FINAL_SEEDS):
            hist = []
            for r in range(1, 101):
                value = .2 + i * .01 + (r - 95.5) * .001 if r >= 91 else .99
                hist.append({"round": r, **{m: value for m in CLASS_METRICS + CLUSTER_METRICS},
                    "train_examples": 10, "ae_examples": 10})
            summaries.append(seed_summary(hist, replace(CFG, seed=seed, evaluation_split="test")))
        a, b, coverage = publication_tables(summaries)
        assert a.iloc[0]["W-Acc"] == "22.00 ± 1.58"
        small_scores = copy.deepcopy(summaries)
        for i, score in enumerate(small_scores):
            score.update({"SSE": (i + 1) * 1e-14, "SSE/Inlier": (i + 1) * 1e-16})
        _, small_table, _ = publication_tables(small_scores)
        assert small_table.iloc[0]["SSE ↓"] == "3.000e-14 ± 1.581e-14"
        assert small_table.iloc[0]["SSE/Inlier ↓"] == "3.000e-16 ± 1.581e-16"
        legacy_history = copy.deepcopy(hist)
        for row in legacy_history:
            row.update(clustering_geometry=(compact * 1e-6).tolist(), raw_labels=compact_labels.tolist(),
                       **{"Calinski-Harabasz": float("nan")})
        legacy_cfg = replace(CFG, seed=FINAL_SEEDS[-1], evaluation_split="test")
        legacy_run = dict(config=asdict(legacy_cfg), history=legacy_history,
                          seed_score=seed_summary(legacy_history, legacy_cfg))
        legacy_suite = dict(runs={"fixture": legacy_run}, active={"history": copy.deepcopy(legacy_history[:1])})
        rng_before = copy.deepcopy(np.random.get_state())
        refresh_saved_clustering_metrics(legacy_suite)
        assert np.isclose(legacy_run["seed_score"]["Calinski-Harabasz"], 400.)
        assert legacy_run["seed_score"]["Calinski-Harabasz_valid_rounds"] == 10
        assert np.isclose(legacy_suite["active"]["history"][0]["Calinski-Harabasz"], 400.)
        assert legacy_run["seed_score"]["W-Acc"] == summaries[-1]["W-Acc"]
        assert np.array_equal(np.random.get_state()[1], rng_before[1])
        refresh_saved_clustering_metrics(legacy_suite)  # idempotent refresh
        passed.append("Small SSE table precision and completed/active checkpoint diagnostic refresh")
        assert np.isclose(np.std([s["W-Acc"] for s in summaries], ddof=1), np.std(np.arange(5) * .01, ddof=1))
        try:
            publication_tables(summaries[:-1]); raise AssertionError("Missing seeds accepted")
        except ValueError: pass
        try:
            seed_summary(hist[:-1], CFG); raise AssertionError("Missing round accepted")
        except ValueError: pass
        for phase, invalid_cfg in (("development", replace(CFG, seed=42)), ("final", replace(CFG, seed=42, evaluation_split="test"))):
            try:
                validate_run_phase(invalid_cfg, phase, fresh_suite()); raise AssertionError("Leakage guard failed")
            except ValueError: pass
        passed.append("Exact rounds 91–100, five seed means/sample SD, missing-run and test-leakage guards")
        return pd.DataFrame({"Check": passed, "Status": "PASS"})
    finally:
        torch.set_num_threads(old_threads)
        restore_rng_state(old_rng)


SMOKE_RESULTS = run_smoke_tests()
SMOKE_PASSED = bool((SMOKE_RESULTS.Status == "PASS").all())
display(SMOKE_RESULTS)


## 9. Sequential development — validation only
BASE is the source's adaptive/full/original-rho/noise-included control under the corrected evaluation and restart protocol. FIX1 must exclude noise. Try geometry, alignment, prototype EMA, rho, fixed split, warm-up, summary smoothing, then only three neighboring HDBSCAN settings. Cached identical runs are reused. Every research candidate uses 100 rounds on both datasets at alpha=0.3 and seeds 7/19.

Promote optional changes only with a mean validation W-Acc gain ≥0.1 percentage points, nonnegative gain on both datasets, improvement in at least 3/4 paired runs and the declared M-Acc/WC/ARI/noise/K guardrails. Prefer fixed split within 0.1 percentage points if guardrails hold; adaptive requires measured superiority. Noise exclusion and positive rho are correctness constraints: record their validation cost and block freezing if the resulting configuration clearly regresses against BASE. No Cartesian search, no best rounds, no final-seed tuning.


In [ ]:
def development_scores(cfg):
    rows = []
    for dataset in ("cifar10", "emnist"):
        for seed in DEV_SEEDS:
            c = replace(cfg, dataset=dataset, alpha=.3, seed=seed, evaluation_split="validation", rounds=100)
            result = run_experiment(c, "development")
            rows.append(result["seed_score"])
    return pd.DataFrame(rows).set_index(["Dataset", "Seed"]).sort_index()


def compare_validation(candidate, reference, prefer_paper=False, require_gain=True):
    if not candidate.index.equals(reference.index) or len(candidate) != 4:
        raise ValueError("Development comparison requires the same four dataset/seed pairs")
    delta = candidate[CLASS_METRICS + CLUSTER_METRICS] - reference[CLASS_METRICS + CLUSTER_METRICS]
    rules = SELECTION_RULE
    reasons = []
    gain = float(delta["W-Acc"].mean())
    minimum = -.001 if prefer_paper or not require_gain else rules["min_wacc_gain"]
    if gain < minimum:
        reasons.append("insufficient W-Acc gain")
    if require_gain and not prefer_paper and ((delta["W-Acc"] > 0).sum() < 3 or (delta["W-Acc"].groupby(level=0).mean() < 0).any()):
        reasons.append("W-Acc gain inconsistent across seeds/datasets")
    for metric, bound in (("M-Acc", rules["max_macc_drop"]), ("WC", rules["max_wc_drop"])):
        if (delta[metric].groupby(level=0).mean() < -bound).any():
            reasons.append(f"{metric} degradation")
    for metric, bound, direction in (("Temporal ARI", rules["max_ari_drop"], -1), ("noise_fraction", rules["max_noise_increase"], 1)):
        for dataset in ("cifar10", "emnist"):
            ca, re = candidate.loc[dataset, metric], reference.loc[dataset, metric]
            if metric == "Temporal ARI" and re.notna().all() and not ca.notna().all():
                reasons.append("lost temporal ARI coverage")
            common = ca.notna() & re.notna()
            if common.any() and direction * float((ca[common] - re[common]).mean()) > bound:
                reasons.append(f"{metric} degradation")
    if (candidate["noise_fraction"] > rules["max_mean_noise"]).any() or (candidate["multicluster_fraction"] < rules["min_multicluster_fraction"]).any():
        reasons.append("pathological noise or absent multicluster structure")
    # HDBSCAN inherently bounds K by floor(observed clients / min_cluster_size).
    observed = int(round(CFG.num_clients * CFG.participation_ratio))
    if (candidate["K"] > observed // 2).any():
        reasons.append("pathological cluster count")
    return not reasons, gain, "; ".join(sorted(set(reasons))) or "validation guardrails passed", delta.reset_index()


def run_development():
    if not SMOKE_PASSED:
        raise RuntimeError("Smoke tests must pass first")
    if SUITE.get("frozen") is not None:
        raise RuntimeError("Final configuration already frozen; development is closed")
    incumbent = replace(CFG, cald_mode="adaptive", cluster_geometry="full", cluster_alignment="greedy",
        prototype_ema_beta=0., summary_ema_beta=0., mdfe_rho_mode="original", ae_pretrain_rounds=0,
        cluster_min_size=2, cluster_min_samples=1, exclude_noise_from_prototypes=False)
    base_scores = development_scores(incumbent)
    current_scores = base_scores
    decisions = []
    stages = [
        ("FIX1 noise exclusion", [dict(exclude_noise_from_prototypes=True)], True, False),
        ("FIX2 cluster-only geometry", [dict(cluster_geometry="cluster_only")], False, False),
        ("FIX3 Hungarian", [dict(cluster_alignment="hungarian")], False, False),
        ("FIX4 prototype EMA", [dict(prototype_ema_beta=.5), dict(prototype_ema_beta=.8)], False, False),
        ("FIX5 positive rho", [dict(mdfe_rho_mode="positive")], True, False),
        ("Paper CALD fixed split", [dict(cald_mode="fixed_split")], False, True),
        ("Budget-neutral CAE warm-up", [dict(ae_pretrain_rounds=5)], False, False),
        ("Client summary EMA", [dict(summary_ema_beta=.5)], False, False),
        ("HDBSCAN robustness", [dict(cluster_min_size=3, cluster_min_samples=1),
                                 dict(cluster_min_size=2, cluster_min_samples=2),
                                 dict(cluster_min_size=3, cluster_min_samples=2)], False, False)]
    for stage, variants, mandatory, prefer_paper in stages:
        eligible = []
        for changes in variants:
            candidate = replace(incumbent, **changes)
            scores = development_scores(candidate)
            accepted, gain, reason, deltas = compare_validation(scores, current_scores, prefer_paper=prefer_paper)
            record = dict(Stage=stage, Changes=changes, Candidate=asdict(candidate),
                Validation_WAcc_delta_pp=100 * gain, Validation_accept=accepted,
                Required_correctness=mandatory, Retained=False, Reason=reason, Paired_deltas=deltas.to_dict("records"))
            decisions.append(record)
            if accepted or mandatory:
                eligible.append((gain, candidate, scores, record))
            display(pd.DataFrame([{k: record[k] for k in ("Stage", "Changes", "Validation_WAcc_delta_pp", "Validation_accept", "Required_correctness", "Reason")}]))
        if eligible:
            _, incumbent, current_scores, selected_record = max(eligible, key=lambda x: x[0])
            selected_record["Retained"] = True
        SUITE["decisions"] = copy.deepcopy(decisions)
        save_bundle(SUITE)
    safe, gain, reason, deltas = compare_validation(current_scores, base_scores, require_gain=False)
    SUITE.update(development_complete=True, development_selected=asdict(incumbent),
        development_selection_rule=copy.deepcopy(SELECTION_RULE), development_safe_to_freeze=safe,
        development_vs_base=dict(WAcc_delta_pp=100 * gain, Reason=reason, Paired_deltas=deltas.to_dict("records")))
    save_bundle(SUITE)
    return incumbent


if RUN_DEVELOPMENT and SUITE.get("frozen") is None:
    DEVELOPMENT_SELECTED = run_development()
else:
    print("Development closed: configuration already frozen." if SUITE.get("frozen") else
          "Development not launched. No validation gain has been established in this notebook yet.")
DEVELOPMENT_DECISIONS = pd.DataFrame(SUITE["decisions"])
display(DEVELOPMENT_DECISIONS.drop(columns=["Candidate", "Paired_deltas"], errors="ignore"))


## 10. Freeze the final configuration before test evaluation

Final-test execution is ready without a development-search prerequisite. If a validated configuration exists, it is used; otherwise the current conservative settings are recorded as **predeclared, not validation-selected**.

The saved digest fixes the method settings and the complete final grid: all five official seeds and all requested alpha values. Test results must not be used to change the frozen settings.


In [ ]:
def freeze_configuration():
    frozen = SUITE.get("frozen")
    if frozen is not None:
        if config_digest({k: v for k, v in frozen.items() if k != "digest"}) != frozen["digest"]:
            raise ValueError("Frozen configuration was altered")
        if frozen["seeds"] != FINAL_SEEDS:
            raise ValueError("Official seed list differs from the saved protocol")
        if frozen.get("selection_source") == "predeclared_at_user_request":
            requested = replace(CFG, seed=FINAL_SEEDS[0], evaluation_split="test")
            if requested.config_id != ExperimentConfig(**frozen["config"]).config_id or frozen["datasets"] != FINAL_DATASETS or frozen["alphas"] != FINAL_ALPHAS:
                raise ValueError("Method/grid settings differ from the frozen final-run protocol.")
        return frozen
    if not SMOKE_PASSED:
        raise RuntimeError("Smoke tests must pass before freezing")
    if any(r["phase"] == "final" for r in SUITE["runs"].values()) or (SUITE.get("active") is not None and SUITE["active"]["phase"] == "final"):
        raise ValueError("Existing final results lack a frozen configuration")
    validated = bool(SUITE.get("development_complete") and SUITE.get("development_safe_to_freeze"))
    if SUITE.get("development_complete") and not validated:
        raise ValueError("Completed development failed its guardrails; inspect those results before final testing.")
    selected = ExperimentConfig(**SUITE["development_selected"]) if validated else CFG
    chosen = replace(selected, seed=FINAL_SEEDS[0], evaluation_split="test")
    chosen.validate()
    if not chosen.exclude_noise_from_prototypes or chosen.mdfe_rho_mode != "positive":
        raise ValueError("Correctness constraints are missing")
    validation = SUITE.get("development_vs_base") if validated else None
    if validation is not None:
        validation = {k: v for k, v in validation.items() if k != "Paired_deltas"}
    payload = dict(config=asdict(chosen), seeds=FINAL_SEEDS.copy(), datasets=FINAL_DATASETS.copy(),
        alphas=FINAL_ALPHAS.copy(), selection_rule=SUITE.get("development_selection_rule", SELECTION_RULE),
        selection_source="development_validation" if validated else "predeclared_at_user_request",
        validation_selection_performed=validated, validation_vs_base=validation,
        version=METHOD_IMPLEMENTATION_VERSION, final_round_window=[91, 100], std_ddof=1)
    payload["digest"] = config_digest(payload)
    SUITE["frozen"] = payload
    save_bundle(SUITE)
    return payload


FROZEN_CONFIG = freeze_configuration()
FINAL_CONFIG = replace(ExperimentConfig(**FROZEN_CONFIG["config"]), seed=FINAL_SEEDS[0], evaluation_split="test")
display(pd.DataFrame([asdict(FINAL_CONFIG)]).T.rename(columns={0: "Frozen final configuration template"}))
display({k: v for k, v in FROZEN_CONFIG.items() if k != "config"})


## 11. Final test — all official seeds and all requested alpha values

This cell runs the complete frozen final grid in one invocation. With the current dataset scope this means:

- CIFAR-10
- seeds: 42, 101, 2024, 3407, 8888
- alpha: 0.3, 0.5, 0.7
- 100 communication rounds per setting

That is **15 experiments in one Run All**.

Completed experiments are skipped automatically. If a checkpoint contains an interrupted final experiment, that experiment is resumed first and the remaining grid then continues automatically.

A checkpoint is saved every five rounds and after each experiment. The evaluation ZIP is refreshed after each completed experiment; on a catchable error it also exports the latest saved state. To resume after a server/process interruption, rerun the notebook from the same working directory, or set `RESUME_FROM` to an externally stored checkpoint.


In [ ]:
import csv
import io
import zipfile
from IPython.display import FileLink


def export_json_value(value):
    """Strict JSON: undefined numeric diagnostics become null, never invented zeros."""
    if isinstance(value, dict):
        return {str(k): export_json_value(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [export_json_value(v) for v in value]
    if isinstance(value, np.ndarray):
        return export_json_value(value.tolist())
    if isinstance(value, np.generic):
        return export_json_value(value.item())
    if isinstance(value, float) and not np.isfinite(value):
        return None
    if isinstance(value, Path):
        return str(value)
    return value


def write_paper_archive(suite, destination, include_restart=True):
    """Stream archive members directly to a path or binary buffer; no loose export files."""
    refresh_saved_clustering_metrics(suite)
    runs = [(key, run, "complete") for key, run in sorted(suite["runs"].items())
            if run["phase"] in {"development", "final"}]
    active = suite.get("active")
    if active is not None and active["phase"] in {"development", "final"}:
        if active["key"] in suite["runs"]:
            raise ValueError("The same experiment is both complete and active")
        runs.append((active["key"], active, "interrupted_checkpoint"))
    if not runs:
        raise ValueError("No research results to export; synthetic smoke tests are excluded")
    frozen = suite.get("frozen")
    if frozen is not None:
        if config_digest({k: v for k, v in frozen.items() if k != "digest"}) != frozen["digest"]:
            raise ValueError("Frozen configuration was altered")
        if frozen["seeds"] != FINAL_SEEDS:
            raise ValueError("Official seed list was changed")
    seed_rows, final_rows, completed_final_keys, metadata = [], [], set(), []
    for key, run, status in runs:
        cfg = ExperimentConfig(**run["config"])
        if key != run["phase"] + ":" + cfg.config_id:
            raise ValueError("Run identity and configuration disagree")
        if run["phase"] == "final":
            validate_run_phase(cfg, "final", suite)
        elif cfg.seed not in DEV_SEEDS or cfg.evaluation_split != "validation":
            raise ValueError("Development export contains an invalid seed/evaluation split")
        history = run["history"]
        if [row["round"] for row in history] != list(range(1, len(history) + 1)):
            raise ValueError("History has missing or duplicated rounds")
        if any(row["evaluation_split"] != cfg.evaluation_split for row in history):
            raise ValueError("History mixes evaluation splits")
        if status == "complete":
            score = seed_summary(history, cfg)  # recompute from observations, not cached tables
            seed_rows.append({"run_id": key, "phase": run["phase"], **score})
            if run["phase"] == "final":
                final_rows.append(score)
                completed_final_keys.add(key)
        metadata.append(dict(run_id=key, phase=run["phase"], status=status,
            completed_rounds=len(history), config=run["config"], manifest=run["manifest"],
            classifier_examples=sum(row["train_examples"] for row in history),
            ae_examples=sum(row["ae_examples"] for row in history) +
                        sum(row["ae_examples"] for row in run.get("warmup_history", []))))
    datasets = frozen["datasets"] if frozen else FINAL_DATASETS
    alphas = frozen["alphas"] if frozen else FINAL_ALPHAS
    expected_keys, progress = set(), []
    for seed in FINAL_SEEDS:
        seed_keys = set()
        if frozen:
            for dataset in datasets:
                for alpha in alphas:
                    cfg = replace(ExperimentConfig(**frozen["config"]), dataset=dataset,
                                  alpha=alpha, seed=seed, evaluation_split="test")
                    seed_keys.add("final:" + cfg.config_id)
        expected_keys.update(seed_keys)
        count = len(completed_final_keys & seed_keys)
        expected = len(datasets) * len(alphas)
        interrupted = active is not None and active["phase"] == "final" and active["config"]["seed"] == seed
        progress.append({"Seed": seed, "Completed settings": count, "Expected settings": expected,
            "Status": "Complete" if count == expected else "Interrupted — resume" if interrupted else "Pending"})
    complete_grid = bool(frozen) and bool(expected_keys) and completed_final_keys == expected_keys
    identity_columns = ["run_id", "phase", "status", "Dataset", "Distribution", "Method", "Seed", "Evaluation"]
    def identity(key, run, status):
        cfg = run["config"]
        return dict(run_id=key, phase=run["phase"], status=status, Dataset=cfg["dataset"],
            Distribution="IID" if cfg["alpha"] is None else f"alpha={cfg['alpha']}",
            Method=cfg["method"], Seed=cfg["seed"], Evaluation=cfg["evaluation_split"])
    def scalar(value):
        return value is None or isinstance(value, (str, bool, int, float, np.generic))
    scalar_fields = sorted({k for _, run, _ in runs for row in run["history"] for k, v in row.items() if scalar(v)})
    client_fields = sorted({k for _, run, _ in runs for row in run["history"] for client in row.get("client_metrics", []) for k in client})
    warmup_fields = sorted({k for _, run, _ in runs for row in run.get("warmup_history", []) for k in row})
    report = dict(archive_version=1, method_version=METHOD_IMPLEMENTATION_VERSION,
        cluster_metrics_version=CLUSTER_METRICS_VERSION,
        publication_ready=complete_grid, configuration_selection_source=frozen.get("selection_source", "development_validation") if frozen else None,
        validation_selection_performed=frozen.get("validation_selection_performed", False) if frozen else False,
        completed_official_runs=len(completed_final_keys),
        expected_official_runs=len(datasets) * len(alphas) * len(FINAL_SEEDS),
        official_seeds=FINAL_SEEDS, development_seeds=DEV_SEEDS,
        seed_window=[91, 100], std_ddof=1, accuracy_units="fraction; publication Tables A use percent",
        undefined_json="null", undefined_csv="blank", restart_bundle_included=include_restart,
        exported_runs=metadata)
    with zipfile.ZipFile(destination, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6, allowZip64=True) as archive:
        def write_json(name, value):
            archive.writestr(name, json.dumps(export_json_value(value), indent=2, allow_nan=False))
        def write_csv(name, columns, rows):
            with archive.open(name, "w", force_zip64=True) as binary:
                with io.TextIOWrapper(binary, encoding="utf-8", newline="") as stream:
                    writer = csv.DictWriter(stream, fieldnames=columns, extrasaction="raise")
                    writer.writeheader()
                    for row in rows:
                        writer.writerow({k: json.dumps(export_json_value(v), allow_nan=False)
                            if isinstance(v, (dict, list, tuple, np.ndarray)) else export_json_value(v)
                            for k, v in row.items()})
        write_json("run_metadata.json", report)
        write_json("frozen_configuration.json", frozen)
        write_json("development_decisions.json", {"decisions": suite.get("decisions", []),
            "selection_rule": suite.get("development_selection_rule", SELECTION_RULE),
            "development_complete": suite.get("development_complete", False),
            "validation_vs_base": suite.get("development_vs_base")})
        write_csv("round_metrics.csv", identity_columns + scalar_fields,
            ({**identity(key, run, status), **{k: row.get(k) for k in scalar_fields}}
             for key, run, status in runs for row in run["history"]))
        write_csv("client_metrics.csv", identity_columns + ["round"] + client_fields,
            ({**identity(key, run, status), "round": row["round"], **client}
             for key, run, status in runs for row in run["history"] for client in row.get("client_metrics", [])))
        write_csv("warmup_metrics.csv", identity_columns + warmup_fields,
            ({**identity(key, run, status), **row}
             for key, run, status in runs for row in run.get("warmup_history", [])))
        # One JSON line per round keeps high-dimensional geometry export bounded in memory.
        with archive.open("round_diagnostics.jsonl", "w", force_zip64=True) as binary:
            with io.TextIOWrapper(binary, encoding="utf-8") as stream:
                for key, run, status in runs:
                    for row in run["history"]:
                        diagnostic = {k: v for k, v in row.items() if not scalar(v) and k != "client_metrics"}
                        stream.write(json.dumps(export_json_value({**identity(key, run, status),
                            "round": row["round"], **diagnostic}), allow_nan=False) + "\n")
        seed_columns = ["run_id", "phase"] + sorted({k for row in seed_rows for k in row if k not in {"run_id", "phase"}})
        write_csv("seed_scores.csv", seed_columns, seed_rows)
        write_csv("seed_progress.csv", list(progress[0]), progress)
        if complete_grid:
            table_a, table_b, coverage = publication_tables(final_rows)
            archive.writestr("tables/table_A_classification.csv", table_a.to_csv(index=False))
            archive.writestr("tables/table_B_clustering.csv", table_b.to_csv(index=False))
            archive.writestr("tables/metric_coverage.csv", coverage.to_csv(index=False))
        if include_restart:
            # Serialize live accumulated state, not a potentially stale checkpoint on disk.
            with archive.open("latentfed_revision_state.pt", "w", force_zip64=True) as binary:
                torch.save(suite, binary)
        archive.writestr("README.md", "\n".join([
            "# LatentFed paper evaluation export", "",
            f"Publication grid complete: {complete_grid}. Completed official experiments: "
            f"{len(completed_final_keys)}/{report['expected_official_runs']}.", "",
            "- round_metrics.csv: scalar metrics and losses at every saved round, for convergence/quality plots.",
            "- client_metrics.csv: per-client correct/total counts, accuracy, loss and routed model group.",
            "- round_diagnostics.jsonl: one JSON record per run/round; client order is selected_clients.",
            "  raw_labels, aligned_labels, routed_labels and clustering_geometry share this order.",
            "  assignments is indexed by all client IDs. Raw labels < 0 denote HDBSCAN noise.",
            "- warmup_metrics.csv: separate warm-up rounds and AE budget accounting.",
            "- seed_scores.csv: complete 100-round experiments only; averages over rounds 91–100.",
            "- seed_progress.csv: completion status for all five official seeds.",
            "- run_metadata.json: run status, configuration, package/device versions, identity hashes and counts.",
            "- frozen_configuration.json / development_decisions.json: frozen settings and validation provenance.",
            "- tables/: final Tables A/B and coverage; present only for the complete frozen grid.",
            "- latentfed_revision_state.pt: restart bundle, when requested; extract and set RESUME_FROM",
            "  to this trusted file through RESUME_FROM when restarting on the server.", "",
            "Accuracy values in numeric exports are fractions (multiply by 100 for percent plots).",
            "Table A uses percent. Noise fraction is in [0,1]. Undefined values are blank in CSV and null in JSON.",
            "Filter phase=final, Evaluation=test and status=complete for paper test-result analysis;",
            "phase=development is validation-only. Interrupted histories describe the last saved checkpoint.",
            "For final statistics first average rounds 91–100 within each seed, then compute the mean and",
            "sample SD (ddof=1) over the five seed scores; never pool 50 round observations.",
            "Clustering seed means use finite rounds with explicit coverage; publication summaries require",
            "five finite seed scores. Per-round error bars for curves are a different aggregation and must",
            "be labeled separately. Use original raw_labels >= 0 for intrinsic clustering plots/metrics.",
            "The archive contains source measurements, not rendered figures, raw datasets or external baseline runs.",
            "Synthetic smoke fixtures are excluded. No unexecuted experiments are supplied.", ""]))
    return report


def export_paper_results(show_link=True):
    if not EXPORT_PAPER_ZIP:
        return None
    has_results = any(r["phase"] in {"development", "final"} for r in SUITE["runs"].values())
    active = SUITE.get("active")
    has_results |= active is not None and active["phase"] in {"development", "final"}
    if not has_results:
        print("No research results to export yet.")
        return None
    # One accumulated archive for the complete server batch.
    path = PAPER_ZIP_PATH
    signature = (CLUSTER_METRICS_VERSION, str(path), (SUITE.get("frozen") or {}).get("digest"), tuple(sorted(SUITE["runs"])),
                 active["key"] if active else None, active.get("round", 0) if active else None, ZIP_INCLUDE_RESTART)
    if globals().get("_LAST_PAPER_EXPORT") != signature or not path.is_file():
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_suffix(".zip.tmp")
        report = write_paper_archive(SUITE, temporary, include_restart=ZIP_INCLUDE_RESTART)
        os.replace(temporary, path)
        globals()["_LAST_PAPER_EXPORT"] = signature
        print(f"Saved evaluation ZIP: {path.name}; {report['completed_official_runs']}/{report['expected_official_runs']} official runs complete.")
    if show_link:
        display(FileLink(os.path.relpath(path, Path.cwd()), result_html_prefix="Download paper evaluation ZIP: "))
    return path


def run_final_experiments():
    if not SMOKE_PASSED or SUITE.get("frozen") is None:
        raise RuntimeError("Passing smoke tests and frozen configuration are required")
    verify_final_runtime()
    frozen = freeze_configuration()
    selected = FINAL_SEEDS.copy()
    plan = [replace(ExperimentConfig(**frozen["config"]), dataset=dataset, alpha=alpha,
                    seed=seed, evaluation_split="test")
            for seed in selected for dataset in frozen["datasets"] for alpha in frozen["alphas"]]
    # Validate the complete requested batch before any training or checkpoint write.
    for cfg in plan:
        validate_run_phase(cfg, "final", SUITE)
    active = SUITE.get("active")
    if active is not None:
        match = [cfg for cfg in plan if "final:" + cfg.config_id == active["key"]]
        if not match:
            raise ValueError(f"Resume the active {active['phase']} experiment for seed "
                             f"{active['config']['seed']} before continuing the official grid {selected}")
        plan = match + [cfg for cfg in plan if cfg.config_id != match[0].config_id]
    pending = [cfg for cfg in plan if "final:" + cfg.config_id not in SUITE["runs"]]
    print(f"Official seeds: {selected}; {len(pending)} experiments pending, "
          f"{len(plan) - len(pending)} already complete.")
    try:
        for cfg in pending:
            run_experiment(cfg, phase="final")
            export_paper_results(show_link=False)
    except BaseException:
        # Preserve the original training error if emergency export also fails.
        try:
            export_paper_results(show_link=True)
        except Exception as export_error:
            print(f"Emergency ZIP export failed: {export_error}. The saved checkpoint remains available.")
        raise
    save_bundle(SUITE)


def final_seed_progress():
    frozen = SUITE.get("frozen")
    datasets = frozen["datasets"] if frozen else FINAL_DATASETS
    alphas = frozen["alphas"] if frozen else FINAL_ALPHAS
    rows = []
    for seed in FINAL_SEEDS:
        completed = 0
        if frozen:
            for dataset in datasets:
                for alpha in alphas:
                    cfg = replace(ExperimentConfig(**frozen["config"]), dataset=dataset,
                                  alpha=alpha, seed=seed, evaluation_split="test")
                    completed += int("final:" + cfg.config_id in SUITE["runs"])
        active = SUITE.get("active")
        running = active is not None and active["phase"] == "final" and active["config"]["seed"] == seed
        expected = len(datasets) * len(alphas)
        rows.append({"Seed": seed, "Completed settings": completed, "Expected settings": expected,
                     "Status": "Complete" if completed == expected else "Interrupted — resume" if running else "Pending"})
    return pd.DataFrame(rows)


if RUN_FINAL:
    run_final_experiments()
FINAL_SEED_PROGRESS = final_seed_progress()
FINAL_SEED_ROWS = [r["seed_score"] for r in SUITE["runs"].values() if r["phase"] == "final"]
display(FINAL_SEED_PROGRESS)
print(f"Completed official experiments: {len(FINAL_SEED_ROWS)} / "
      f"{int(FINAL_SEED_PROGRESS['Expected settings'].sum())}")


## 12. Publication tables — Tables A and B
**Table A:** classification accuracy in %, mean ± sample SD across the five final seed scores. **Table B:** intrinsic clustering scores on original inliers, dimensionless noise fraction and Temporal ARI, again across seed scores. `TABLE_COVERAGE` reports undefined rounds/seeds; `FINAL_SEED_DIAGNOSTICS` retains inlier/outlier counts, rho statistics and model-group counts.

Tables remain empty until the complete frozen grid exists; incomplete runs are listed explicitly below. Full histories are in `SUITE["runs"]`. These tables make no claim about unexecuted baseline experiments.


In [ ]:
FINAL_SEED_ROWS = [r["seed_score"] for r in SUITE["runs"].values() if r["phase"] == "final"]
expected_rows = len(SUITE["frozen"]["datasets"]) * len(SUITE["frozen"]["alphas"]) * 5 if SUITE.get("frozen") else 40
if len(FINAL_SEED_ROWS) == expected_rows:
    TABLE_A, TABLE_B, TABLE_COVERAGE = publication_tables(FINAL_SEED_ROWS)
else:
    TABLE_A, TABLE_B, TABLE_COVERAGE = publication_tables([])
    print(f"Final tables pending: {len(FINAL_SEED_ROWS)}/{expected_rows} official runs completed.")
FINAL_SEED_DIAGNOSTICS = pd.DataFrame(FINAL_SEED_ROWS)
display(TABLE_A)
display(TABLE_B)
display(TABLE_COVERAGE)
display(FINAL_SEED_DIAGNOSTICS)


## 13. Final accumulated evaluation ZIP

The ZIP is saved automatically after each completed experiment and is stored under `latentfed_revision_outputs/`. Each refresh replaces the previous archive with the currently accumulated results.

The archive contains round-by-round numeric metrics, per-client evaluation counts, clustering labels/geometry, seed scores, configuration/environment manifests, progress information and—when `ZIP_INCLUDE_RESTART=True`—the current restart bundle.

Tables A/B and their coverage information are included when the complete frozen grid is available. Partial/interrupted runs are clearly marked and are never included in publication aggregates.


In [ ]:
# Final cell: create/refresh the accumulated archive if needed and show its download link.
PAPER_EXPORT_PATH = export_paper_results(show_link=True)
if PAPER_EXPORT_PATH is not None:
    print(f"Final archive: {PAPER_EXPORT_PATH.resolve()}")
    print(f"Archive size: {PAPER_EXPORT_PATH.stat().st_size / 1024**2:.1f} MiB")
